# Crawl corpus ViBioMIR trên Kaggle — tự upload Hugging Face

Notebook tự chứa code crawl, không cần clone repo hoặc upload file Python.
Phần chẩn đoán đã tích hợp: Output tự tạo `<run>_diagnostics.zip` để debug.
Bật **Internet** trong cài đặt notebook; dùng CPU, không cần GPU.
Chỉnh ô **1. Cấu hình**, rồi chạy các ô theo thứ tự.

ID bắt đầu và kết thúc đều được tính vào khoảng chọn. Đây là **ID gốc**,
không phải số thứ tự dòng. Giữ nguyên ID/URL nguồn, không dịch hoặc sinh lại text.
Trích HTML/PDF có text; Selenium thử lại URL lỗi, trang JS hoặc text rỗng/QC lỗi và render JavaScript. Chưa có OCR.
Dùng chung `priority_probe.py` với notebook 97 hostname: header trình duyệt,
timeout/retry có ngân sách, giữ lỗi theo từng lần thử và kiểm tra challenge.
Tài liệu nguồn: [ViBioMIR](https://huggingface.co/datasets/AIGuruTinix/ViBioMIR).

**Bản tự upload:** `crawl_corpus_kaggle_hf_upload.ipynb`.
Mục 1–9 giữ quy trình crawl/xuất của bản gốc; mục 10 tự upload output đã hoàn tất
lên [AncientWings_R2AI_STAGE3_ds](https://huggingface.co/datasets/AncientWings/AncientWings_R2AI_STAGE3_ds).
Trước Run All: mở **Add-ons > Secrets**, tạo secret **HF_TOKEN** với token Hugging Face
có quyền ghi vào dataset này và bật secret cho notebook. Token không nằm trong code/output.


## 1. Cấu hình — chỉnh ID tại đây

Mặc định chọn ID **583–1582**. Để thử trước, đặt `MAX_NEW = 10`;
sau đó đổi thành `None` và chạy lại ô crawl để làm tiếp.
Giữ nguyên cấu hình tải trang/browser và phiên bản code khi resume cùng run.
Với khoảng lớn, chia thành nhiều run nhỏ: crawler hiện tại nạp danh sách công việc vào RAM.

`SOURCE_PARQUET` để trống sẽ tải nguồn đã chốt từ Hugging Face.
Nếu đã thêm Parquet vào Kaggle Input, điền đường dẫn `/kaggle/input/.../links_corpus.parquet`.
`RESUME_RUN_PATH` tùy chọn trỏ tới **thư mục run đã giải nén** từ notebook này trong Kaggle Input.
Để nâng cấp run cũ hoặc thử lại lỗi, đặt `RECOVERY_RUN_PATH` và chọn `RUN_TAG` mới.
Recovery dùng lại raw 2xx đã kiểm tra hash, chạy lại extractor/QC, rồi crawl URL còn lỗi.
`RESPECT_ROBOTS = False` tải trực tiếp giống `domain_samples`; đặt `True` để
kiểm tra robots trước các request trang. Trạng thái robots không được chuyển sang Selenium.


In [ ]:
# ===== CHỈNH CẤU HÌNH TẠI ĐÂY =====
ID_FROM = 583
ID_TO = 1582              # Bao gồm ID này
WORKERS = 4
DELAY = 1.0
ATTEMPTS = 3
MAX_NEW = None            # None: toàn bộ còn lại; 10: thêm tối đa 10 URL
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36'
PAGE_CONNECT_TIMEOUT = 20.0
PAGE_READ_TIMEOUT = 45.0
PAGE_REQUEST_BUDGET = 180.0
RESPECT_ROBOTS = False    # False: giống domain_samples; True: kiểm tra robots
ROBOTS_ATTEMPTS = 3
ROBOTS_CONNECT_TIMEOUT = 15.0
ROBOTS_READ_TIMEOUT = 30.0
ROBOTS_RETRY_BUDGET = 90.0
ROBOTS_FAILURE_TTL = 30.0
BROWSER_TEST_MODE = 'failed_only'  # 'off', 'failed_only', 'all'
BROWSER_USER_AGENT = None  # Dùng User-Agent thực của Chrome
BROWSER_LOAD_TIMEOUT = 45.0
BROWSER_WAIT_SECONDS = 15.0
BROWSER_ATTEMPTS = 2
CHROME_BINARY = None      # Selenium Manager tự tìm/cài Chrome; hoặc đường dẫn có sẵn

SOURCE_PARQUET = None     # None: tự tải; hoặc '/kaggle/input/.../links_corpus.parquet'
RESUME_RUN_PATH = None    # None: mới/tiếp tục session; hoặc thư mục run trong Kaggle Input
RECOVERY_RUN_PATH = None  # Run cũ để reuse raw thành công, thử lại URL lỗi trong run mới
RUN_TAG = None            # None: ids_<ID_FROM>_<ID_TO>_synced; đổi khi muốn run mới

from pathlib import Path
import json
import shutil
import sys

if sys.version_info < (3, 11):
    raise RuntimeError('Code crawl yêu cầu Python 3.11 trở lên.')
WORKDIR = Path('/kaggle/working')
DATASET_REVISION = '0148f6f80ffafed5c005af6d506ccfd9d3fb47a7'
SOURCE_SHA256 = 'dee1153f5b9d9010dea18e1b7724b4915ff847b711d036bec8b933486802a971'
SOURCE_URL = f'https://huggingface.co/datasets/AIGuruTinix/ViBioMIR/resolve/{DATASET_REVISION}/links_corpus.parquet'

if type(ID_FROM) is not int or type(ID_TO) is not int or ID_FROM < 0 or ID_TO < ID_FROM:
    raise ValueError('ID_FROM và ID_TO phải là số nguyên; ID_TO >= ID_FROM >= 0.')
if type(WORKERS) is not int or WORKERS < 1 or type(ATTEMPTS) is not int or ATTEMPTS < 1:
    raise ValueError('WORKERS và ATTEMPTS phải là số nguyên dương.')
if not isinstance(DELAY, (int, float)) or not 0 <= DELAY < float('inf'):
    raise ValueError('DELAY phải hữu hạn và không âm.')
if MAX_NEW is not None and (type(MAX_NEW) is not int or MAX_NEW < 1):
    raise ValueError('MAX_NEW phải là None hoặc số nguyên dương.')
if RECOVERY_RUN_PATH and RESUME_RUN_PATH:
    raise ValueError('Chỉ chọn RECOVERY_RUN_PATH hoặc RESUME_RUN_PATH.')
tag = RUN_TAG or f'ids_{ID_FROM}_{ID_TO}_synced' + ('_recovery' if RECOVERY_RUN_PATH else '')
if not isinstance(tag, str) or tag in ('.', '..') or any(separator in tag for separator in ('/', chr(92))):
    raise ValueError('RUN_TAG chỉ là tên thư mục, không phải đường dẫn.')
RUN_DIR = WORKDIR / 'runs' / tag
SOURCE_PATH = WORKDIR / 'data' / 'links_corpus.parquet'
RUN_DIR.parent.mkdir(parents=True, exist_ok=True)
if RESUME_RUN_PATH:
    previous = Path(RESUME_RUN_PATH)
    if not (previous / 'manifest.json').is_file():
        raise ValueError('RESUME_RUN_PATH phải chứa manifest.json, records/ và raw/.')
    if not RUN_DIR.exists():
        shutil.copytree(previous, RUN_DIR)
    else:
        print('Đã có output trong session; dùng run hiện tại, không copy đè.')
if (RUN_DIR / 'manifest.json').exists():
    if (RUN_DIR / 'crawl_config.json').exists() and not (RUN_DIR / 'corpus_config.json').exists():
        raise ValueError('Run cũ cần nâng cấp: chọn RUN_TAG mới và dùng RECOVERY_RUN_PATH.')
    saved = json.loads((RUN_DIR / 'manifest.json').read_text(encoding='utf-8'))
    if saved.get('id_start_inclusive') != ID_FROM or saved.get('id_stop_exclusive') != ID_TO + 1:
        raise ValueError('Khoảng ID không khớp run đã có. Chọn RUN_TAG mới.')
print(f'ID: {ID_FROM} đến {ID_TO}, bao gồm cả hai đầu')
print(f'Output: {RUN_DIR}')


## 2. Cài dependencies

Lượt mới dùng phiên bản tương thích mà pip chọn trong môi trường Kaggle,
Phiên bản thực tế được lưu cùng run.
Khi khôi phục run, notebook cài lại dependencies trực tiếp đã ghi nhận.
Nếu bật Selenium, bước crawl tự kiểm tra thư viện Chrome trên Linux, cài phần thiếu
bằng apt và mở trang thử trước khi tải URL. Kaggle cần Internet để cài package/driver.


In [ ]:
import subprocess
import importlib.metadata

package_names = ['pyarrow', 'requests', 'trafilatura', 'pymupdf', 'tld']
if BROWSER_TEST_MODE != 'off':
    package_names.append('selenium')
requirements_path = RUN_DIR / 'notebook_requirements.txt'
if requirements_path.exists():
    install_args = ['-r', str(requirements_path)]
else:
    install_args = package_names
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / 'logs').mkdir(exist_ok=True)
installation = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', *install_args],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace',
)
with (RUN_DIR / 'logs' / 'pip_install.log').open('a', encoding='utf-8') as log:
    log.write(installation.stdout or '')
if installation.stdout:
    print(installation.stdout)
if installation.returncode:
    raise subprocess.CalledProcessError(installation.returncode, installation.args, output=installation.stdout)
versions = {name: importlib.metadata.version(name) for name in package_names}
print(json.dumps(versions, indent=2))


## 3. Chuẩn bị code crawl

Ô dưới giải nén snapshot nhúng trong notebook vào output; không tải code từ mạng.
Các file `.py` sinh ra có thể mở để xem. Snapshot có sẵn của run được giữ nguyên khi resume.
Snapshot code được lưu trực tiếp trong notebook này.


In [ ]:
import base64
import hashlib
import zlib

EMBEDDED_FILES = {'audit_snapshot.py': 'c-pO5TW=f36@EA1|1hCj?6M_SmpDdCkun_HL5#$S;W$MBh1iUChvY<i^X#r9Mk3H64}B`oAO(t`2oNVIf;1?aqJ5Gc`k)~HL;WTF&dls`MUk>y)PP0q&biIGd~@co)oQ&v8bvY@OeTdGD_Km}f)rnqf`yWglQb`6kTafy_|M8D;=3Y>M97rbm*Rls6CnyVN)_WQQ2Zcbs!Y6At9AXFjI&e~jH_|RRW7by8>uvAK??SukZF=LOK~?Xp;6Tw<Bw%lKTLQ&iRAA5xrigibN%UyJm8tA*FMVA#Hz`7L2Q+ZM|iBs6n2xw^T)Y>e45wa%R~lgC_-NF>({K*=n%>qhSbE_lq;1Uc-ZOF5)OfZ*;8V>ek~LusNkVXCbfIluQB{nq9|3O4fdjxhq+$$h|Vv>Vd07d%4IShIAt;F{=#Xu=hA{S&UjF`M7SW(VUG7vDigP1y_cja=8^nZxXwFo+;uukGkSRz$--?4&L`X@js>AM1Rj<*U%O=}yoopr<yhp9Fc%PWsIItm)D;`l8Q|Sf<1$8SJ<=;~i0~<yQ7#O8KTHGPZ#P*z9)>=*Jg(F2rey)^m3Uf8CBnfIRf?+_Op<nU?47nf&BOqC-eItnmr*g;SYNjSz?H*PD@T8jq};XZP>p3uBbQ!#c(yAsPvi)Gz_JcAQLZtkdhJn+9ZM^Xk1Lv-$5|vaMdY+i^mnI)fYvs9Wl)RWv6Q}vfHSln)-mET7wlI&D#d-IpmLq}G_8&ri)9}3BA8SL7&$O?r;&PjTNg0uF=1DkwmdOS9ubDX?4eO34l~p^P-4K2)Q%%Vn9T5{RFN~($`mOxN@muyv{V6^mOZuV_--WJh7UA>GwiUDjDQ#UV8<oV9p=ace<O)xtTeaV!?stXem2#w=l;La9<5<UuCL_0h^NC;<Sa=Grd?fa<dK1b%JWJDsR}8_b_^OAr)Ng5f~q<#EGb+&j?!J%`I*h3%Dbrjl1I<+traq>H*UEadt;!DMB-XfopSVC`|!%mKQ0sKAuNhSAxIV-8~}hE&{?Z^a4+9t2VB}SB8pfr5y4&tj4JZVHv!Lu_RO%E3O>~?y*{#=SlnB~Jd!|^<PPl(atjiJoHF+gUOa^laLD<2!HY7dHE%3{6$|g%ARR?~OcXoA*<2AEX*qD%`2^Zd(@pemHw~w-g-$Uq6yzW!^1O;JY^gbi3pIaHjs*)6r$b{omterkAdOm&`N2gb=+#*>Yh708P>nm<$NJ1C13<3d4n1A_!h(~1{gU=P0n5u6L}f)D@DLbP<OdP}yzBh*(Y-gimmIH=CkY6MO^D$Ah~uR2E6{S<l&2QCE;AQOP!`e2?$LRl7Ya-|0y>h|J0x|eLbHeiixxv>!!J!X3!SmlT4~9i1Zcr1xeqhQ*S68>icl7*>SpM*wCi1A%!l!LZcTRT^Llt$C2C2fFQ@V_5Gwm0)di72FF2_C5bJH7Bpu>yuQ0K}Q#UHY(^+H<bKCO`m|ZOsM%7-O2filrg9(mo;GaP-{ca_r;h?G%YF~t!%h(64W^?eBc_SiZ@H=TDmTak_p+Yg=C|fHTOOkVLg29%d$Foc%p=(V7yFV(UXxcEYy;QMDvK1;&C^FEeydm3JA6{jP^`ky+5eM|*wI8h{?~vfib-cATO|>+_mje^?@$&`Bc)U^^nu<N0*4$*+M2V5pwzm!VxsqnGoSPiKKzrrJc~S6SLcOAo<H2OlPN~J0B;Kee#1~#JxC$n&az60fEF2x9TRpsed>G}2x7+UR-cOEqZr>QTk^Ut$sNREBnjww+pfjY+`fTI1b)7$}J+~srSLBlB)>UO&<=jsL<(p>MS>fv&>_HL=l0bLA-6#$Au#u5aKs=xd-F|TTuzT-5?65@aqp=^0L?}*&yk0FP!WIvy_LghW#T_+cH_rrf891hI#WWK1MUE7Y^*3BZ_3~2Cufad^LX~DdL7%%~pUvK4#~wTsc;|z}B2wWI?q?Q?ip50JU9irynsA9K!j0ZbTHwAlIr~>KX2ID%SuW_74E?X4pFBfT`H;&kwkOX|KY^ze*RdF%pMDyU-&0n#6kfY=tdXVj!*CAI7rU88Za<U(5H}qa59mIgh~oZX7Ntt4<2X;dCMxu;Fkkzxe6WLjk)z|jezASr$NhNQUNCJPEI46#%BG`{giBK8$6W1|nU6f$TQqceqeiWW!-GI%$VBH#Z7_<oEeJBJ9khAPI9}j{6K#v0gY&}DB2QlJi5{x<mI|40%nmx-k<P3@9~HXwrN@zeoC~(Ca2dlTQ{CH|oS<E53^BJS>Qt*HyjrOmdMw#wi(wU786R~kM^Y2g?YVO)Lv@P@Kr;A2jT+Enk&8L=*^png$)UutZnoN9A{T(EujRQ<J)AkU!G-;~feFasv5Xos*D5-G7d-Z&8XDrBuZ#Bf@_L%bXMR!3l9RQ=*4u2Gt-pYtJV-MEfzF+>NMjAO2+kT1xH+2J(k`O=7ER95D8zRFgr2Z5o8z9dMEs=0)!H5kelvsC?RBTRj`9-ysc#d}UPZkFj0(*m<LXI5SjHJ935KYHHassCCMbCz<=_ntFzjHjbhHdM+qAqeL|CTLlv6c<Xee|nP&X8-Wh&e3=Ni`M7f0*sRr)I!nb*6kF5Nn>!lFSFi=byuOa~E<cS8VneJ!mA0^0MK8WJ>^nWXnvf4Da{(SPgmb}MlmT$mCwd$OD~)#{k8%F?teG%{1!0jmcJngP=;>&|*8K)Iud0gjlH<ZctM;qX|uU(df*7nlHv`EA-f68_Y8-QMMUb7$H42ryIdbOD+%Nu=U$Kpdt_xxemXva{J(q_#s(pcdDbYKU>+3V7rvWxR_q(glpL5{<`%n!kakba^|@teFyZ3>*EPap?agVRZ;rCoK7h0_5~x0V>gaQ(s=U`+7<qaWJ&@Wox@1ivsg`x+95V(9$ySR)=Z(4pkctDV3FZf6csG1PsTX@N^YE8sV<i+qjh-(!3l7c1yX1vLDN&w|O%=+@eTyCqj-VAoFg`LMWF}ezc475vsRAjA<l8b|VOatqen3?5wx3305nHz5|Cahn4Cml(>rWX%D|ngj@8d3!QjVh%SCiDB~pWZHyGd|E)34v<WkW^ypv$?{#xrH$@N6-2=t5trTb3C`u3XQ;#Rpg9%#PEwiPY%`MG5I@yk(B2~Q`zr6M8tZy3ACC-^_Y(JXF^OJv<%^nT)#)`8qB`y-@r@t#}c<HZ!zMpMBJU{uaWXafkOyt>LF$xrbpPQQ=yL*26c>ytJ&%l57rD^7h^V9FB1xM3&_Eo}mNjU1M!vam6P0mlgHsmIp9{b@BXJ1Vb`5(XM)$$(fEO0d70So6Re~F3x8xBi;{Qms(3(0=?^z2zOVPkpr%<HdZ+s^RP0Itsa23}yCZ>J~rhZ(Q>eWs$Ie61R!QLAqgW4mw9jqQ38#QyV}Ke8jRyB83}Ocy8+mH}QgR2GV8E!*y6;KB4bW8hE<InZE!b3$djx=jvD#^w3xXBd9WJ=w8tLR<Y=%<P1ujbObBHA)yo&E{QOHV4IDi+J9a8#9Rw##XwUM%>i_!tllDLFLaaSyXy?z;apk8um<otxH=Cgs9tWh$VDbJZ>Au1FbI{w)MS;M6HIRIMhINgS|B`fYkoBXp&Kzj^TZ)i>dJ1`;Xsddz16iKW9}eFbHfmud}tIMwdJDS{JI$=`tE^?pKxZ!>5$HUkA<ld+*IVU5q9dx*yVi-#UE^;=h9z>5m^HN($ZQ8PmXFaDyG$Oxt;0dpU%?SJF~e>w=~0osIg!rpUC{E0SH*CoY5F#NRicJbb(1r&WqB_M*et68RNQ=^m?gd0o!!*;2^bvb8~m)i7be13D<K7lL#5>{$S~Ir}zi#v?iVa>@>ADFNr-5|$~zal&c9K!F6qGYp4GbHZYcXgO+$Mdzn~qT8>%9V2PJ#V6XK35D+Do2iXp0v5@_5{q!&Kk%XZ;S4x{)XhETvZ2R>vtO`d_8uu}?);djk6E|dW%_RqZ-!N>S69gqD2~wN%|LUc8+?2-$2Z<~%+0Uy3y--TdHPg$>uzl>Z-)v*1IdS%df*TBE<e$pd~k1zeYjAz)jIwCeNY43etj4K<xdFKlL;@kK!$z(Hfdr#(WV#V8pXzO!v)m1^ZvaDPyEOCAHDndi5G{fP$LenXkRd1e}Vgh{+-wU%8Z`X_Y(|-d|!9lzE8h7`+jxv$MKD+Ob8o20+M7T#}MSM0xkV}_4RB21)<}oc>',
 'corpus_runner_snapshot.py': 'c-o~KU2ojR@%?_qRzX3YbD|`FH9dz9s;neN9LtsE7{&F#P~_enBOb|cmy%Wa{zLmv^tC}y1ZWYo=%fA62M75d`X`;)4|12}o$N#be0RIEv$Hd^^SRmD+B&W>!WWdVMOf09#Q8EzGE$WJDJ3+UCmCfV%wkehr)k0#<k7?JqNI$L=SemvQC=1mBhi9JXGNZ5oOxSYTZ3d-<RvFzIWNMJ(ZQ_DmjIilG~!8~F`@tt@+#xB9FdsL!Ybu)5^=paV(0p+{792iui{BskLLQ@*DTNU_hrZza=!w;!5Y2SQ=CxW*(K9o`J$v@3}+b(zC1jB_UPyd@rfhPwOv+OM$7H<+s@$O;V1jgAD=vf*{cD8Kc@mF1mT=!+!+(+i~LVXnua^~yj|kHNU}J;VC2aO+1vGY9}pP0fA;}-dH?Q^>=#8!U(nOf6TWlr&TqUs_lf)YXD44g9+5OTqvTWIWj-Va3pfPb*}DS>@IUe_oP}kgmN+9xL`>80687KoM#Kp@r^|vf_<aX{@?=T#3aOji-i7(Q_q-_#pH}gl;^^K-Fe=f2L<^8j0AkO@`cJ}?Q9UUJgS-&3gUKsB)Y?#uMmcZ|kTv`7-BTO&{OMIF24l$`q_~;m<z-M4{#$cy7A9#`(tz_+E?X(GPfHL1h-jI|6eWa!G!F7Cy>v#+v?azTc}8I_61)#ilPoMR^{|aK%`I|)w+7COFyR5Cz=)Ws<2REA({7IjtHEFZ+E2vsS-1q<U~W`~7b)HgKt3FcjhA7T%qZiae`?kfB%toQ07iC*qhUN8bcSMGGOOWdaQ<KsvW141+X?8@8IoW5^)3@N7l2u~eeb?A9oOP3!-P@tWtdj<uq^Y^b)E@Of|$V)gxQ?Np4$B);S18gotGChb7>akc>8>(;<N3KfQpb2#(^8lW=q8uIQj$^i-R*a1=}<vv%DnujeymX>+r}p;=#|PaEDXPhZOd^Dkn}7J5#Z)6pJMCl{i0wsDa_8BR{N)G>Jk^)p9V4OrIH)wJK}8rg%%xh%&aMI|4$|7=K83wJX{#IId)zgupLbR}xB|VI^3(rH_~8T&(lx%Y<>}ng@_1p&Ba^k&s?hLJYO$xDwEVYv<yY!x1q@suJWV_~x2)>9b4(z;qKMXU;+GI)H~e&yOFIWx~LSM+>ksNX*J+tK464RXL>&V}<)t{I&R6(r+qxiPj6mQ9w*hGp9zP`J&MzOz?C)qt%_?yq`7<{o936L>U_?XFP%%mNS|GMsY|_P6vP~0+Zz`%0_zUp)GH8u&%^l&UjT3gEG4c8RH?Zm~Zdw%VkpXa>&th!#?X?vr(Z_za;`WCKXD7EjTZFVT}oA^G+0%DmVl0Yh!4}tsTX159AHINS+0#oc?U9tt~r!ceK_VB^Fa-OaW+BP;5{?+bq*iCdVw$8#!j^rs9INUUPxis;Q6MQrOKWmKs8Qh^iO``<hfsXWjADSseP3mtn+d3>pSbl!J2)_+>$Thr`s)f-lnt=xQLg<i2zq>rSb~V~bgj0{+PqRYJrPEf^UKt>qsOTLOvT&pWEE#@DZfsI6cCqhbWVgg`iUZB4X6fYRL|`H<Lh>VX-NJ44BQZQL))gi|X}(3WPcPtkUMz3G)mmBA<}#?qVYuXd+JUQ}r)lYkA849lxBq5<H+phd+q69+61hwu`T2Ife&4hm2en=sWk3IV0AD|Eua-3V2CQ;-?>uB(;g2!Mm}LpTwZOsy0KLKDU%>oCkvAwm;twAKG6chu(=45u#v#+It#tx2+qHaq1Pn7W+N%dzRh;kS&T;0sddT3ApefC6;m@F8<U{AT$H9H{+)(~Ox$$i4^8H&vJ>{L(<6M`md_ml%Z!(qE+QHhptH;y!0$En9<4?Pephjj@?Cw6I&2)ETZi0?IEYZERDKRf`#$n^TOZoG<r}f``wK4jcg{)Y29(tX$(X`jpujwamSkn<y?%ho+iL9Kad^Kmc-=w+)E$&8sU^N1F|$7E?4nCLZR61%&uu*76HWXKy~VeN=hPChh&E9sFWbhL#Al*-yjeX&jPrOx*3V_fTNvO!8G~psZ(6COO!+MCKh=8g8sgBzPY>7{?tZY2e*{SD=$pK*53}3!vZtNH7bTgG(sZERqaCQqb6_B8ppfV;zD?&cvyiTqiveyP>JgAHry=1sKB19X&AlOQtibhCx4@=Ze*=S}q~cM*AdU1mXF|Rx*Z3d5&M3DU>#jqYgj@*;+uh4WRZhmG&j4DOLtxd$;Urq;FlrnssY1nlaHiEd`Ydu`-TRI|h>X|Ceqn^+)a1=#5y{mfkt$T_8SWn$kNd2&Q<~NK|RY`UpZ(EhQ&?<wy(irtqVxpmIX9V3xd86leqn1=4*cG>ydtT<40qD)!6$(|dTR+LhG)s^g*7d8PK2@93j`wqKgo0ZA4vAv#0Op>|A~a5y*zvXz2*P=IwtYxGY4xz5=l&*s5-m<Mr{M{3cev=vua{TpolbZ8WrR3*oUBCq!h$~f8l(UF_c3#{w-j?=gkp<I|>!^Yl2Fo!S1fGZJ<qypi2nJ;yzy;Q2_0>Up?)jx?RaFJWhwASsA=gTAtPA@rJEUDd+xpIHB$l}(y5JzIAi@LJTwat>bqpV*lYV4@uACQB$-$vwuo-){$L~s9_k@Wh_-x9*h>XJm)uYV{|u5I0Me)@Msc=Gn2qLiN#e*NaJP@KnNN_73^`;cJVjC27WkQPyQ2JCxs{rbBliLYOOpLu<lXInr1`R#Xr_4>^Z<f=BPtDP&;)~;}?2ju+i-*fWgUw-<}^_%Y^a;1#H#5A_k)dO{+W?GprNIkaN=Q^p6-Qa~?AoaU`{SQu3F$4#~9SZOaB!$UEn2@vU*Z<?>n+lFcN@UC&itltue*b*`@uQPJ1jmO@kB(2gW!!se=M{M<`yj8#NyAIUHW<WTS}@2^aN98YSd1L2yF^~~pxxfyCgR_*^#QQW?L;R_L=Q&W!scEGUD0vRZ0i-dLg>b!u1GV~^}WVyPuwA=MS(T0mIZSqqzX2sSnFX}Xb)MGB)(`6faG6onOgiIsxh+kU;(^IgcLqIIe9u#>{?#G{ucO!GJK}&3>LIdWB(<&JSJatP0CkNOw+*)ch7lo_`Bfn`1t7f8ORpb{4sO9uc5jqs>Q5z3-t}u6<ev1x?)>p{Ru6kR<mS2BDY55A}`Nq$^5$`sGPqHz(D!3gP@`S@E;<P6{9KxMoN9rjY6W8s}v|P{<(EaH=1CBTXK9=7xKl=iQU<liWCD=Hz%LXoKdIIF-uCwnrRp<l(P*W^3ZJ);U~fr>bRTA-OsbLEWgNzxv|$Y7_c<sYT{b0r0Ec=q)6gvo~N5Ou@{nM1qD`ULgF&Lmzk*ccVgBxVT|LcqzoH{UczQc270ck2jo)`H5<Rl$3EF@C12e-HfGkVnrMktHcoRe=E?b5es$VH4acx3-q6wmNj@ffn^@XjQPUUJWZ@=s4%bPM*YsM}cckPwVyMbNKqR7e&~*km55$qGiHeaH<xql{V{oai%%I!*6Af0~Wg7Ris)9g#4|-w{M?r5|u17YZ5tn-t5?N`lQLLKQmWG9MfF4F?Zqu-9?sZ^wb5w6RkaIHdHFeWi)6>amwe*^Dm$6KAWVCq3o%43p<Un=t;a2Y`Q>6NM4+ykJL0iDGk>lr2f|LDEg;F0M9vp$B!I8q?)BTgrJR>HfsZm)Ilv;;1N@}8a$GF?BOAftxnxDGPEwy5J!}W6>(|H(OqRk$^E5(H8q$=iR*qE|4S&PBiQO(!xis$<nhN`=2bd{$a$m65m9i0UGpB}>dfc_;OgZ?5U^E+VB_J)()sZ~F*or}#IsjG})Rk#v?*Q`nd9pJZWi|TfF{FN^M260mQ-G0~_QP)UXedsl6`~ieY;Brw|b<!*%DZE*T68B9o_J@#&5om%ENgYbAq#{0BirPo)*(?Yd8<@KkVFFe(%<>F+wkaOH0QBJuGlwt}7O*TU@w@?yBfp?jzKKz-i{^Dp^AaW!HJ)bQ8|u7*taoo#4h(6|OU#+jxPY$IuT{LqXl)E7)Ws11+HKM(AU&>9F^UBObSjOsT2fWb0K1FR+*NTx<8W3<Peni4aoJ7LHYH04$Lim8<TM=A3+&deAX9asTPg$+1;&6#>*K#S$IU)Ro&2F`w^-k`q7B0#=H2T|3q!6Ueob!Q!@AC{^yams{Dv*#-V7U3fQ-9FNPg8+!2-TWFm^xBr7j9LtLD)AYRp(0(%)B5`kAE_TlkP!QG?^~yr%18`lC%u+A+}VQY=l628UN-vox76c+2KAX3lT#U|PVnC5O(R^;oCIX+Pn44|HZY*`d&iHK(@LsC!g*=B4MWEpJzY`c1LUDby!6mf(q3O^JN)1=*O6>FSv#EbJb(hv^oxN&CR{8KH3?o?SJpkgu?MHqj}N`oz0sM)dx=3_-VF^$W8G(0+K<v*kuN1)R;)55HwjARmQWjcKn+WCwkn@g^45UKMN8L2KHzyeMouT61-V>uy+^A>FG58CrfJjA!=&lh!-kY_eVpVA?$NL`#~XS<`Lpz!`435u!G2KRamBOt3jn3KhizQIgGAID@i49HY5(dYTGdeok29PujC^4&{V!R-Js>=3Q&mI%^~^p#pY$k8Iz=7T#ZIuinizy^qc5-^a{#I|!^qxX*HQ6U+Wx)>xwEmNo64cXGYMMh78b21&P^&^rWGKR2*4!*TU`LBCe^o?yAS2qTK2l8+gky{>6jz4s%C!CKF*koH#DY^U9NYi3uigtCCS0dMchO0>ubCDYK#U`^e6PVM1nfG?(Hew(Uz!dxz2pbD(r^>SQdKwK{VqlyWaOii;1qQf?_{9V%{QdkfIeF;=3!sUlmn^UXgAu;rK>z9(Hd|+3LZ@-*FR*~_iV#bXri{Kq79?1`&0v16AZ>;pdZUv!Q;$nCO*^790w8#^Xt3Qz`tTx}htG2gg?MY@VG&Ur)23}sG_gbO?0$3xSLV^C&k52r%8&`ru-Uf$ktlS8$^SEu9ZYfUz5W*A6HR1zTF<1BA^SZ8CS4t2JkeX#%fH8HnsfLtPber;~YJgTag9lEFoO{KkqucWMe<t)J5pxC>RYAWDRE4L5tgpt4XYypBw^L*hrxO5De%MOY@tPRcLEE4m6Ny<PFtW?TVzxC1(fI)UToB;J2?Eg<4gyiX4OH-r*NgEOh!u;GA}}4@H`VF^IbNE<{{hBkD}4',
 'crawler_snapshot.py': 'c-qZ<Ym*#TmEiaK6=^xVStVCjSGOdu-BKB3*?_{b?2%=5Ct5|4U6s|9?yAaiW>!CF^#_Jv23WiS3q!!}OpJ|}m<bp#GsCij#V#M@@c&Q*vdyRc3wvJo=DnFWtE+|B9S5|sGVeY2b<Vlxb<aI_?%WsS7vkiV>$ki?8Xip6lG)_Qo5zzR?|3&8@2y3Y9(j{+Cz@n!Z<-A0-#iWDSv=eIlGKZ5!{`v6JLk@wYmCxl>IK1QkuTCH2)uYYPtx2AXR{;^^EjDhjfQ%c?#{zBi@44pnM|Sql~9$gC5u@erEPB*jl#twAI5`R6`l<iX&TM)4!5N1+#aV<ID9QhCNCXEgGHV|^)L$t$#g!6@@S~*=h0z4iFZ^zc{2@X;ck?2sbQE$c|46&8TA{64}ZUs%%tnlG>j*mMIKLdH)vp3KFE_m)l&oC{YpGn&*Lx~LpRh<{s&%krcoY3#ZZ5lhr=NZG*d6$%#xXUo`(6D`!a_oV_2`@2YF7z*)W;vU%ETl7`v{2C4;>v*S~PlvOFH>X0s#xnA2Rt%Ig=#AQ!WE0LVf^pq?$#366_qMs*L~y_v+bHa=wYNt|0n(_|;fseGy`+`?C%izm@*^jhQ6{3uM*<N%f$W}x)4gNwh&Bl<Wfe+2k|;yHbMtC$>km*vGs({L0|LclZ`Og<T^F6~BnFq#Y-jW51*>-Foe-t>B&zyFNic;hNQDdM^nWl@+8#@CR>qIA96+3*{;UVY)!+rib(zjX8VlCplI(EtkY7Wtsr+G=?4Um8JIW*Q+ov*e(ua1s!4(CWllGD=dwi)Krz593`JQWF=%h9W4~$;RR4rDwQ6XB-_$1yvs=APTT-6LCyH9K`vUkc?k;l6f?1`st3}!U-L5AxB#TOd1%qJ)hh5UG0W>GK~kp&QTs^)Joek#(*zi?ustxx45#%Af|i6IBoKetam$I0Q~~L%JLxDqn}(QkEU|~L^W|#!7Q9c1hRuq-qYU5?;LBC@8Gh>Lx9<Aopkc)+%Ld%4$>GXpL<MwEk32ud=d^K=_`+riz7YD_>!GBWq6LkW4X?7F`Z{kC<kMiWyos7Y!Ju2&xMl=#)m9<miIPWooI$s*z_0qXzf|QrJ9^1;jn1Y#s_Ku3uH}WWpUgAvdAWZgemBifnvpp^b$xbs~>~!caZLz##mC<NLn!M{+*fc0WN#E(=gn+QH>M88(<#xc$$j2(5DU-q7?;@K49D=Ifzn?Z8IdAN&J<liDMN&_z1XRu(VE5&cFG&Ykr%{2uQWa{+$#$-=-m0Z`C^!dKz`I#ZEKzx9{ZMokh1BcKc6bK~JE!%t7hWcV@>I+9wQ9H^#Gj*vaxVo;O=GeTO)G2&co7K^<tdR2don9C`_)(uB=ftzWXp6@}s34*ce>#3QeXAJ;t}_RV+%RNKK7g|QNWWr|%<zrz^#{ujf^B6<lJZ`$-<UF5))y>No4b%d<J3q1wsGK_G68wE}3V4NiIBlKp`fw#5>jdo=6vSg7CBCCJ=N*KWiY5J|sYs}?w=V+SWVF>e4CmW1m4-a@lK-|c@Cwktt9}jUp0c-p|_5ws0;18YrXdY$2V1pP$1`rmSWeB4<nhcv*yG4bU&caKFa}awX*iIOl7Y{*N8hW@aAYVb_C>kxMvrG*+9%ei!CdqI|kEyMS_Nedu7XPD03KcuE#bg4~5C~Q{qWPdh&%ladlb;RpCX{J)Kwz350ouSMOn{%`yJbUoCC)OD1&0g3tpRLHYGJOwsQXPVVb?{jYx3(+oB}Vd`S_&w#;qG(xX8x|Fv%k?&b(|f8pVSc_#2`1JWXITf#xrJ`#_MmC3+(m<dL@%!FTu(<%2PDY;Fyz?1b<JsR0%aIBy!z3t&#g?9rMVgT`vO6i|r@cerS?3oD$!WKCk+v5N9C<A^aa%ngW9GoXBc(8h&rDQp5No+3D)<1dZ#e4c$~YkhtDOY21V1tT;6t4#mMzdoapt0{*W+wzW~_7mxeOuhOK<Uf$xk(H$r#Rk@)sSB;`&A#S@W({7KZLZk&p6+=Y<w0hPY13T9&OUYyh`;4s@iw~MZgmi5@hWV9$%G8-)q9@cS0KX#7G<EBW%#M*CqyN{!zv0VpY&dOYY|S?a8m$IG)V@eq_A9*Mo1QEytANphyz$I<F6xNMYtO=UcI#d{RSo$b(9W@h<njd24tB6X-n^0ZLiC%9*kq`2aJoVSrev5Wh0-g9Q==}R4|R+>Z!a^UN_<>tG1{9vtjDxNrBXRqD^#e(Lcg?qiCo$DG<*06If-Ol0tiHBAp21j-pwW7c8UBR(VDSAb_<MS%Oa0Ttb>{n{GF(k0mlYJC)#I>S<&S+9+-CYEo2DtJEZ(#<<a1vtC|iWXA{&SB@%FR}DXR!Z`@y!{#ya|1CXL#MZVvVnaT+Kwt6U#2yi7`3{UARUd}}>=Y3I{=W;Gk1_gIwa!SY)%IBRdYj7i0sdL~&Dg}G-SdRvnkjD%IP)kq>Z#YbUx<`6pKT8awl@i<JHyaRx4cw#P==m|X^xybK@|Zc|1i%}!g=UVJPhbNAjnNnzW|b}Kr&a~wdh2(a7UxpxY;(T1@&ebOM4xJhIj;fTmqbEZ7m+IF*Kxe`3aDk(cxgS$m0E|rG<x)r`yFH2wN%|%AyGiBX#lxY_&j=utPRQIC{S4w;3V~T)n*F^JKO97MT^zlvnHc{Q1;eS`y?_wGE#XYo$84e7y`};ihPIYdD>2Lw%8?u@rEH#L{B}(g`trR3SxfTU(<&=vae4=@`4N+82&_Y+GKp#Lk&fpV$>Ti#ZB1O~oBybTT)6V=lNuEhVI{s_C+tCW1QyA+ddetHv;Smnzg!RhY0KV3CU}L7ijWcORz)mJc|$GGBqfISqKmkPxP{ydZYMgC%Rq=kQQIhllp(q75x#0&Bp=x=@w2Wq8JQ(wi;46(Iw@@q+<~g%A`_0=uLsowU^ymCDdEF|d=m>9Um~YoZ1~TYx-4m?K|9!w#Z9%vToS8~^I{&o9!&ZLqN(t}C+;QWlsgNAWI}7$oU@k>NYqq3{(MWqo6Wyk77Lq!=M#AnGic>_@Qawe^CE^<~Sj@raUS0>fzlIT~AnrBVdqJ6Q_tAD$nfaK$sHS+Lnb#XA9-Fe>USaa+YV1I@(fTQbLfgQNk!wA4Ybm@^0#2qKsJ?;NX(6pYzfhSVkkz&w;VKdG=mx|k{6UvM7Kv-jg<ky)CnF5fcjX7CF6@pj*me$d#dd4SE(k_qu|)xbLT1F!{%1)fLej-e_u%M%*wIhWD`FdRC3p%lzOTU<moAEid1<-)cEP+Dc^T#P$_Tv_)9MSAs-Rc`f_eyyMh`M1bqaQ($~R2rvoM!HqUujvDt@W56X9t3EVLW?DAo}1mScco{QK*bEvG$(TFqUd$OrPKf<d*0P(bQAqbot*?MA3gF2bcPWcs?o&Xgc`j2veB&Nl&Zu|iXdozGn*4E@4JPqx|zcfAVt*99ZSap=rP!l(s017w{_FxMFDdHT0Q)J@qQG~fDiC)UY$3dYuWMu{AopysemznidM^xCgE;?svA@SA%bdFO9798#f5<|@vpE{T;q&ZWlKpQKMQCe^CRRvxX5jRdfJGY3k0ZF_K=q=B(u0VSnkqf0O&*eN{@DxD&XoiWpD>1d$fP4m2>qAd!qCUlbDOLt|pMNRrW*8VHY#mq?`luW%#_`^AF*<b1;(SrWyITZX>7@5}=3DHrh2$;ME9ro3X56qcNC-S>}-|hH2%c!$CAhgMctbw3-I6fyH?cfGRW@kvt5{0{xabcm{i561GHaFc(q*6oBcW__&y(ZrRcB7y`0JE?M;Gb7<Cz(%L@2Xs`p5@Tj-Z0T~JQCGbB0u3l(+=g~e7szVm&x41pEtIR@*0bS2vQUmlc0pQ-oB?KuSSxt~H7rOAJ77iG%)1zQ#F$A3NJ=ZmWj9{2({Lk|V!6?rd4Tqj+O!BO{uXa&fLK$^2VjECt;80UP;?7X|D!LepGo_EFs(}oz>4M)<Qc=tnS7#qA`<0tPm#a({V6fUjllVNz;vA$SWk5E0w%Hnsj-*_dfFb@ny47~>9mRjX8#$0fogY?(XW~-<%zV2#qypVcNX6S#AsOgd05V>lSi%{orfrW54WhG_76I*A)68KnHf+UI9jsPl>=AQu^>Y&z9XVDK<}F<eqXED_F+zSICFvXY2&9|tN@nF_7{-Z4G=@eplESz;Op{dtRBP5?f@@LTu&l+hz8JHQ7AaTpYDv3|-9<<eYvI+s0nrc5Qcu7ryq8JaGmv|LK<u8F1^5+G*-6{@zjJJ0LWt8NDqCqWoLK#$H|V5O+#cHu0M~@ur_nB&s*lJlV&D~7s*ACfxQucoa$V4*a<SDm34_biVl<zDm;qZLwQe*g+5^atqy`m(Aq9=9_$90_%5<aznMyhSqVu}6(>kRN#0`W)Fia_d>DYe*)XlZ4yC9wc!M<@7I&gJ>8#gJ+u!`&Bd^%}^97Dzg0I@zFjxN8|>vo=NpI<*uPnZ0YRyl&9+BVP1XDL)>04m}*8R~gbE~%#N%;E4LwlEj2X>%I61p5^u*+4wjMNsD_XOZ73ypmMfbZyHkkd@NXt2u}d2T27PhyfUKjvuqzgv54N-zfS;^^D{P?^)Q;FJ8RRItyqR4OB+9HbiPfs77;7dRJkIj=Vfk?k(g%(LM&HDIp)?)bIs+`aD{jbr1~q(G>_%>KuIGxwvT*heifWyqTh%P|w5Ys(3a^nx(mHTf>E_O&DNxg?c@;C@d>H+ilgj163SXC!PoL*u?H^U+iyp`xv<s>`s!Ma8k#l&52yhvTzhBIDWf~2?oguO$7;VAIvq#piV}4B`WeVp63WRl;`-x5U2}mKT)q_#UNs^aJ=}0yG2h$N3rV@8F7$-?la3LM=W7`;#9LMBX>jTSg~Kt=`=G2=oU_@9x92=o;<A82?uX20ACDV5=~@LtnX^*XoKt)WB1l*QaaNFY9tT}0u1p~M2A(G19C&GlR!aqzr`!z@iql&<2Zy(7WlzI2>U8{*9;3W_RJs$VAFP!SOsQE`fBb`WW2z6BAbPy^JVNbr%>DRklLt?=X=X$VijCH+g$7%sVNwM2sBI%W+ht$bs?KX(Y%QrEPX7`lXm0awp!}EL&;@QDFhkyg!KPx8>~cbi;QCiYb-DrGfDxp*0fR&ofXJxrRumvxW_c}4wFyKoPZ%dYN8=ugwdjd>)s|i*t8H_s_w8$Xh}uM7A?YHVjPMjOHe47A_}Kse?kBKfj0eQDufIiIBEjilQtMHX2?`>FzHXo)CwfyEIFcBBq%LTMFPrRYB7B^pSg7D!ZWS<)!{~-=Exd?)v7IviX|0)ewpjEe%%w;711MwPPvi)3P!`^<8TK4EQEIA6&6P#I50K(%R5sd9EGvazl)cMHfniDrK-VCT4x<s6`$+7&5NrHR!b&)83$a}`#%{WdLn&mM5`I4vNnkb+*U?qjh|c~z}UrhAzB0JECv)i-?;{Er|VF1#(Le0F&R#!wt&S53Kh1=bL#Gm;?}Y$nI;?h2hpTky8>buLy7X@=5uZDQuo3V>|pL<j8tg>WVgSCy5QQ?5l|;ib9V`nIJb)>qf$Rb^%fnw9)3hNLqok@-rA|%S=uG%*#qff2cKb*r}L(k3Fz~Z&Km=079(hTx54HmkFNrwftfg<pRLpTjw*;7mkj}C8-<s?W50%=U^UvBt%@}@7hN>$Ff?#WOk9$Js~Z}fT>@6y)ypLWP)xV86C{wPV95r`wAFqvn6UXk*?H4=H=dcnBKQ?GD|BwqM+~A~=SrRYF!vjlKty_Itw^L<Vo#TTy4FBbj0rU{u-8*S90&RP$K08dtz(UK!3>OGC%y{jZnd_zHl8UwFW?G*Fm{UWJ@&aaXz%=>gM##M;SDug_NuGCpvzYt-Bv$eu|_C~B7iT3P<HxDqyY^VLt@}J$rpv`W}bj40aiIeB7i=#>ZB})z%gei#G3X5h6F_klBJ+Z+azaDkkv-pZ99J2+o&p<XWOsTJS*b~4WQ@WVpA6NvnnOZID<V1+Uofx76i*gNpsXFwN*W$u<PwLT;1C)El9tx-8br*uqW0U>^zw_{bCGOc$Q&(knt1pLx4J-dWr-GK`;|@ANZTw==sGE2i&o3`vk}!5s<STSIfuOl;si;_;dw^BT(lBQq&2W&_2OF79x?>3Y^B^34S7+=6*t;#+-t$=2TG8&`C`bYE?AZ^p%uF&AT_0ooJUrD^JzIPTdFV#2%&+)7YU$ztP>i=$)s3+NbVOr~apu5{K#Au$X8ex3)nUfNAzTMK)6+S-rf03!uo^h+pWbNi!YwJxgl3#$VQOcSo_wZ>>00TDBE%z2m(tZ=d}k;0-~fRSk*z13A^6il>F|kGdpH6?d?fOQ%&Fx34(%TByW8BvE*3ovb)4#3~7==$R20ttDdrf$~QR^IwNoV60tNn<2B0V%x*G9RFDDPA5TlEW~*dq{l#xC!O7|__gg>=c|;n6k{4NzJMZ2bX*BXYfKSKd)?M4_+v&x(Dt8FTna|yfdlBUPGl5OQM<`e|Au#4t061zJ=hl10eRq3x7%+EOaK;;^SejxRxALXax`-WL2J1d=GNxx#J|!sfYdg}_=618<r%~v^oeUPfYCV=Vq$O@E54L#s^ON%Qpd^kG|#dqRXS7(vuT_aj|GhJih^}ywR@G-UqxNJ&;<$QLU$AYT);p79{)VcQB@X)xuJ-2iK0+!k8(t~VB!L6yMWVQqW!xf)#^o!`pEYhecqPsP?=XklWhGQ)sOfa4V?lJ%%a4c$z;|<ml-Doaos3dtT+Nme2$J6`n8MYUqP3buZ{tNtPP>B`aA=jUA*!H#8eSh)s$LBD@q<PWGbW-?d+k}&tfd8OzszMzjDJv7dW7Rm#AnB(rH>XrwULchfb8vjW@N>wX7bWRc{t&CUGCnYWbd?0IWbm#|H;~mFyhdi{Lcd=(e{0bVJy7kVv&3K)@Gk?Jc$IJK_MzV+M8NpN(_?W-Rtek)^+I`}S*|hSoX@eHwzlH88N)gEfgz{8EY5p6Pb~Uk<lG=GDi`CKhvWtXWodn;QMbsoI=b7phUtR^9EDRtz6i_?sQ@0w3DOl(i&;yiR>~N6iqBrk10{w5odtma5kodh2u)YL-H72(|{TMG^>7c>M+gJ$)m_&9GSuv)tX*iV`uhZZk?*dF5R?+V<K;|5OfE!uhJW<>n^KR+HTJ#oD94ylM@vVLep}sM3?un6P|fsxewLPJy7M;X*=nIr@w48s?l{&uPWkT7l9^4s|ULEpw_%A$z%Z!oglL!h`Xf-ud%hQ}t~PEJ<Kal7pR)>;MyppjWxGd8t+BbQWNLr0e;SJj^C`<3fUc6xJt_#l0pL;&nA#mccT^OPN*;T>hEtQ8rbne&Apnx&pqg=moV-?6RVuv8kt@KrONrlU}V-BBonxzFu1xgkYx%#&Ir=({kjKwr@(;!~$qUn2<!m#fk_MGwi80;z<nExV;|FC8p*C&cefwS(Jl1Z)~(GN@9r-sc<IS(Ia-iUd42-EboCJxML=z@kV9hhK0G^j`?&u5U)5Q0f}ssd7FbcXQP&jw{VL!ehY@V&4Btv7){|hkbyVcGGk?hy`BrOYbWO79HC_qCt1vEPNS3<o0@<czBbpfO>Py5qCvIer5XYH$8f6pAWmbp9eyA)@YKyoR0o*`p}fo3fStc^7J=A-VkqgU8FtHf#_0{im?Rr69ujHRS+T&)B>7(o6lygd4}uxKAW9@qKq%N2Q$@QV;Syui?{OMyBTAWz8SSaZMj6ph3H}nXMRRg8ekg%FVkxv7IV)P*Ae;qc4s04J!dw|a>h-EPqi`6~tt?i{PRO2a3hM^+(rk=qt!tb>CmNMH%DRqktxD!_s0y+Id{9+Nl8)*N(yRDLskE#c88MB@l`To?=oQji5l@~G<%p-%%s>;%z*M{)bK-~6VauTKxI9~00H=#W4MoL*7s#!4N2g#Wx>T{6^)io<hQJE4wviXlSe-c7d?otW*&T)I7XFGGwuO=`?$u@xEN|HSM(x@)hAdo-ZN*diE+VTp-&QO0W25XUtM#ESah{&td$!C`rn<VShpg7&^+?Vxadjz=7o%mXvBc7CDv|x_@w9R@$O?M)QtQ96uRFA=3uUd!z{s*PRj)<Wvq`{Dh=Jb=6hF50IS5{)#GZ$PMxEzRq;^uFqyu#UQ8gy$viXgzR2cKr#Xp-A{e*umvi7WoePxwVKk3~_Kt^V7mkyT!VH9Ejm7;_}^&zBo1<x3r_hhH_V%AwjOnY?w(;WX9L@@|5eCBLXuNVSF_IFT`v~oxRcRDakYlW2Xmi1841Y3W?rLQWL#3`0}Y#pwlBGQ%it;goo{o_5k4wo$KR=eby4YPa=(g?>Rw)+;o+SW-$*Lm#zC4KDs(cDTj&qZ=2WD?f?kSFky+dx+{Gtw;?hGQo#q!6yOYNiDeeyngdSFEW-br1>oIt^?xl{B6c*s^CT#U<o!ei@Kj9kL-$6VB`o9mUu|`8iDEh;yHzN&sQRU{i<(lV}YRi%mmd9ph+`QnFym1TB~Zr^yO|ilEWyIG>I&4dZ=6ftb!TZP4FM`g?)?UZlU5=<hT5Hy0oNKza8GA|_*r{F6fb+XqWz4))ujDe^t(O%A7+O>v6j&ft{VKAxWi!D&KfIzZ6N-m92gFv58s#@I$^R6yQ|Lb{BblW8{U)BR!l9e~O?|6FJ9igO*|u55p*Qmc|^4CA_>7*SU?fTW!{%|(7RAp(m>DJaGGG)s)n`$TnU%JY|Zp#U$d1msTL0%O`v>X+-{!YWEAa+&Ihz(K(DEUTppwHkJOeT>PKX9UEq&&x^$Dy)LucJQ>k<1C8RZc)-bw>hyT%m}h{%-&tkZ`ID;>C@FHMx;p5Oz>(Xn3bq&2BMSQf?abAmds$j*ilCqDcZLQM<Swvs&ApM6l2yDuMzC^Gfti$jIRx;RZ&tv2pLO15Ppg|iZJI6Qobf?0f1zTO?3|E6za0Ret!G2@vwJJaejyr$<OtRykmIaLDi*%B^#CW&}flp*fCM62AqvkEp>ELIrwCwH5x~`B^4ueN<-pO_dS_pwGBXPc&vjsrKssVHI%3qHsc|%4M^L0bNBu`N8X3u{qSqE-8<R&E%<-)&hU7ny?L^3{Cn}F)%?s8cZMx1m#=h{0H4}XQ#B~({|3e}r!Z~;wT&S&{!_#Cr>5&q{W&C|CA6?}f}awVZnkcto2*BGyH9lwxfM-8jP&T3dpe~v41`3Oj}CT-gBV4tOcG!k0h5sUnQS!pOl(ylo22WXAL%6guuI`@6BX$Y6~DG=X+-B^hjapoa?nWyTH9fi4S*cO874WZ9%5dRkymzTicpGJ?a!NBm*f)qz-M+bNi=KTEz_pK^aj|7q_f7kKY{=<m_%!s@N5^Ra80O-0yLzlD(4lptw^1Aa-WvKtXDF8(-jZUI)Tun#sslGzPODogZP7Ka%a$;<~i?NyI@mm*&LHwf0kf_0T8~@Ujhry{gOZpZl1l`*1|_@jQ9aIw*6T+yFMznS<En|RcKNdM7Nq9H8l^QE}P`9a%6*iPWf4Aa&KS1>D{<~|64b`KYQcqP5%tRi-?sxEr^yh?Y8BNyJ;^E&C@W2d}zpYp$|%)*#peah{IE=n5HS6KvQT0B&vb}On$A7xz9*(mE=DeP^uk1dZ~z=wiMI+sP>hD&ZXq~pZ&Y5o~#|&DGVBi9>l3BV6I5yqIlY;gi$%b_q1bgY<9zJ55pXB=RP<%=ooJddL<(T-7r$KbwLKg3{<6aeZOLv)phM<SqAdaaCiq2;O}A8648XS?0~n#$JWj)!>WO!G1-~)=*qj3YlkwY139u%G&B%A-d-9GmB2A=>*6G}Tbx@)-mk4IsGc&;>l#u3MKgHl*RAO3qawDl1ZiNwzOZ2-jX*i(Sg<Wqnxi!R%xJb%7vRXf*<}dOsH*2a^@8)>@=S5Ja8}iN@f+e#8xlX`J0MB&HZXiaG7Kh_;H?meKG-KEO!9K6?ggOtrYd!odg<z-)RVEOb2B9P8N*vrlR{EeYnZT_Muh@X?fSOr2Z&wgbYKYq-1eUo3J1?ZM-v*3xMPz8BuA{WnZaxi%*ZC2G2R$#1_lw=qYH=}x1oRM9OT9oSEV(F@WTKsI(sA$=<H%pt+C>f#UYg@SxGUNaWqX#`Nfl=F+99(fi~th_vSTmrg%cx5|HLB8ri(3!)JZa(9vxApZvFfefaf%`S@Eu`NLPg_kVu-UvcH-)ervalb^kVuUaivI9YkaUXh!OH|V>C9-^e^Zpw^Q_9&OeUB#O+Z^D1te=_#MB%;-Z3XicUUQ_#-Xj&-B6$!6eiKlR_7Zq1dodR*PbU^7|!%1$v$KN%FpQ`el-m47i0-T)g`YmB4Rh7!lp=la#O<4=(o&vrNtlaURdhNx}tqJjzau_j%7{IEr+xiNuDHxnbcGQf#<eQu%y#jX-lU9;t*~eDMvjYz{zRhu(aZ4^XJbXpBB@t3##&32KFh|F)l!Sda>bo%aSmPyjN+;l@1>!HlLE7H(y_R~`sxf=pC@WKdAhDLUk1-guXM@%MD3}XcsjX_x*<pETG%hNyHxr*j@FhOewMuMtRk8X~5Eo#3YtxF?mvSm%-#e|sR%2-zAy9278GB$IF0s*4-I?)Piry4bDc~5wBt%~lk#Gd}_UJ3)z;p^KESqXEqb#N~0DCL^Gx0wE!sdmIO<sM#qiaWL3|GDlSThk&awig9g$9-uc?$-X9{hj%&bd2&|1-_)I|pn1r(1rRexT-cttn8*6j$Feef^Kn3Qz3DGZ&t{*w<$;=*B9t!R`X`MSja{UtO044GHkhF4#Kecn5{{n2dCHh^=aVjsAC4pJb5T9h^99Yj<Z4|2bU42V-@O_Mpthv)v0_-VG5!U=y7$)EN>nm*W^9EGCt7G(}js(fC$<2BHdBH7%+x8L3kU+A(%C<w1Ufuhe>xYDhAH8%eUan7dL5RQjreezl{)xT&wB_-oNa;;TIh2ubMnselb)JzrAN(#>y*%Wztm(>DN0UOq%V!H$CB7B+rQS170?!Eg#lC9pT+EIF&ParK&5d)Xd9gCoq4RDs-ka-=B!Sg<{SWiiN>i#Ha9@JpF7+6>W{RAP!~anAHXdkqQAjJ%W5eNE;kc~fphagRXZ3e+*s#$IoPXT8u|RhtG@FbA}~mF0LcOOTC4UDgSRWC+#+NzGnkG{Mqz6%b$LWuRfJ6p|Hd!;4b{mv7dZx(<g^sck9I=iJuNnto$viKlPM<1CfU)M(|PLIq4l$@g+XWaVUn5!Te6CP!;c$brPx<IZlf-`St}tIjb>$Y}bhpUFnKdlqtHr<VIks!H27*yV8hX7e1$=q${kvy$jh`?kTu3KT+)4wAaS_`FQFF)R?M%AB<^V3}a$KVw6-V%e)GmW2}OH1InVO8igmN2*sy^dNB;GNdrW``j`H&ht?Ujc28Y?@1Y{9LfL@YT-*$Af^v;Vu%YpEzMFEmeSF38}+fBUg_G$wx{`DPvI`BE}Q<c4WXjb^l<jE0{~K~BG^CUEBAdFRF~!hchmujbjrxVKJ|_ZLHjFooYkbBR+1E&*(#EH%L<aQj4tKpaj_K92e^u|R=ZODZ&@73x>15GE7oiCVgfP`=#WTPi4NU1OM48?^t#*`QK@5=)~Q>ihAD?jak<YRz~LA%f<*)eCdH2q{F0Nvu3SXzq2q^A-C_=W#}yr}I(Qh0+K`zwuUIJVKy8bI1u1`^&bhX|lDt9TMGbWoLsmq8Aj{AiK1=#<QkqXtp8PwD3m3ax9}i~^@DIFrZZsP4)vp6}99g-@>8SHmLTAvGKsp*ifjUFVwSY0j5Hcjk96)u=*?Lab33f0%dOi;29<Ty_wIZJR#W7^oZK|1lo2r})*0)ZnvXpseQ$iV`+i>d-0^5N2HYyk4m3T8$OpAu`gL;YZr~|~>x}tx*PT?Ki81veCcwhTv(8c0RrLCiwmUt&6U)KZ#n+2*_v{hi#RoYP$g41t|7b5IBn+3+`ny?sO83))JD-v|puTYx&?GL|p|HmJG_x?}s|LUW6Kl<f|e{ugWKK#!8Z{7d?NAG;}eqVXK-GKLF3$V!{%wiyX0)v3ld)7Hn$UPZP;VK?trZ}yeU79gn(daqP-~~DEi-tvaF}4Q^LKjjni!l{jd82%|Q&)AH&?&Z&P~y~@mJwam+i(^&Y-p<B#jX|kgiblU2Lmr=9iy0qemX%Hh~0%;$r!&$0in1X>%xkoYfzObFD7&brVs10A>|<>(0q-05U}21s_4W%>MBk7lG#Ay--c#`1uFpb?RGcn0Eq68Ei$~gt$<LdFSfQp`+=~gu4YYR;Z5C(nm3J|%maH<+ZUzdAVzbnmd)dV#6c*aSO{1MJV*fnc6-Rbk{t@^r2r|kzhu*7FOqAZ+I+_uCK8q)J&5>@=&YRSb!E~KcyP?w|M=J6eE73(Km5(_9)9;*4}bCA?|=Ia^6-aW|Ml<h{ovzo{q?8sfAf>?{N*R#|0jC;?R%em|L-3D%|Cqd!S5b^{q0YG^_z$P<sUwM|Ht_4;~)O#kAL`gpT7P6!|#2S?BxIJyPy2*M-Sis?Wf=PX;p0UfB&0bK79KJ4}bIz4}bAP0{!3Le)x@FfAY^i{Pb79dia-LfB1{v;M)(sgP}FvZ2aJZpU)xC=E1$Mc{}(2aR!OL7x0RL0I!Yj{~TUq_uqxZ*m>~5dx<xNZ^k0yI`WX!V$s?auVg-mb}rN{+fD)|(3BRWibraPx#D#$UaEtD32M0@9-fI43z2Dauz9duo6CS|LwkY)g-xR*a)LTQVBHL68220`gVaVniz3`|SuNPQ^3{Hx##f2WAx~J!6>B>JfP@63!YO?q0nL*0$wg6;{97wAU%1TWQg=D)GgT~Gsw)E*zeP#iw_#ElVy6Kf$N-Viy{x90^Obl23Rg19uj4g-a2NuvA)kQhGQB&^k~NAcrKg-9uRL*Q8`_FTcdyWk=kHztmHm0y+Qr&gH|B&wYLV3;kuByx0HHz(K9WDjDEgExe(=B4{LB`lM7ax@NbaH-3s)U+j*z==;!PxQLwthF{w`LTk!e{5J-qvecmKy9{`SYe|JS!4{``l)q2%|+fAiCizxlr(e&d&Sb36s)?TzpM3OVb8d*1_2@Zf`Y<GbqB{ePSSQ+@EkPmVk)+w!Y(XZ2VFVkPa#M6jO`5CQG)d<4#zdWf>bn_G{P1qBq37jP}sUCI!Mj2d3U707tEAgo0+qrl`h!~KwV2?7R(Ircy)zx#)ue*4q+zxv5LU%fjB=WyIDw3nlHJ$MB0Kd3T+ZGTs7r}|c^LVwUED3atQBZp`TCC4M#ARMrfxTyM8!#>T#TF0AwSy#jI`IHotEq(r}q$_PPWVGo40cuChtYl_Xk{*kATUtOYvL+!)2}j;CWeN;VH`(=;V)LaIEY55KLY^+>*W&6)IY3Lhu^gY3JWefe6|n}!=Q6<$0EvDej~k~pP04O8E2}SD#M^VRj4#Lh?fj?ZtfqQSeMAS8bl8r%G&7@R^_R0}(xnccwJdIGtX>|6nZ)Q`!Jc(#q4k}VWt}!p^5v<D;L7Ub$zu%&j~YWxJR_XQOoFHL-x1VRli-!E+|#!x_Ef7ziZu0LWW(dH)PbJZ+>WXTe69N>%LncpslTeeI;vtGTXVuPT7GO7HZHlSag_@ipY(32152!Ez}?j#d|`ae`WToxIRjBD@rXr=t(Fc6su(U*StcMCcN(kR_Nazq8aiYm6wj#~x~-g8w8tjLvWiqTa+X`?EVN>)>+>Qxp@Jh;-76hfC&hbk*0!33=VZ$}h8HKkWNB+Ps*v&CE-{Dqge9z;Hbmz#th?cO`Ngs}Bha@^;VYBL(cQPg)A(kI;znG3RfM`b(!O8Tb<eVvcTTzB7j1#6=630J;QrFP*ZKBx;zO|Qz#cbLH=)Vl`(&;{Lh%l@t<tr?bhnmVwACu^i<39FU0OG4m#^xx(@Iw_-{ZR(YcA8XuV<{eCyy^?bYFdk08Z4gKi)WGA3)bJKQ$=>F0G%siV>gny@DR@SLQ^?k_}Oy%jrTipKh*5FyHqVU%K`B^;d7=&c&Lt;|Av-pyz;Y(Z(j#<<Q)IH7v-O&r;ufyfG^Xi;Ke=?mGc>5i3K(`SWU5v@ZN}0GS_weliP+ab`zZM}8?k4j<$|eUr{_aT)c~1rxL_2}n7fZ;3|@EP+Mbj5L68Dmp}^S%?VO+EEB-e!z$AY`3q3#k5X%-DSrX;$nG4?l&YtZ0PI4V4qZ%Ns|OyP%wTYN>q!#jPQs@!|4FI$x>+VvWHg}hp=-T;NfMDw*owE0-KD4pW|5*IF~ZRFh)hI1~+3VL3Kfq@&TEB4VOHnapvfKx+Daq&dFK$>SlIxiJ_;b<U7k;%w}N`!wNgOalbeA!VFmfErG5xE9QO0yS*vGMAfdj+B*yPBl`mHqDu8>E*7r2##_EU;w|3SicxBou;emtYRq|;H_BVGLee86rR|dI&6jowvpb6|((Fvtxh2c0uS!l11hF{3+AaDAZaD#z7=zaW0tTdp-oS`6fk6>WH3vo#EsZYnw7q2(WE4_Ia(mu2G6|}4q!i+Jo9=ad4UWLPbQld5Kr2nGjQJ4ov^PPM9~|WeB@JbAL5ax`UGYOFTkK5nRHo8bFz-oOVKGr!dTnP|f%A0HVOwBR>cZJ#8c}|k5Sb{OqlZmt9Qb@WaR);034jhq%#?49<T=?UY*Z3qAFpECsNBPf-SM9CHZP%hs_Us9;x7vGla;FCj^jdV-TMHYG~oZ;S0~K3)VO_2pfX%c^sUvZ0j;m8C_@}U`I2h!Edj4b34^R}otq#a^1%;GQLkkwJUseY68aBw5Po2@NJ$U_#z|SxnuU4^hjU(iop0iU(|P-ldp<)o93&rlhZ`>EHj6QZ9^F&|;eCKj(~yGy;L=7)y6(xX<RHMO;2xln>ytmG)<=i}9F0>FQRX(kKw8>GC@9qrpWfl{Q`i88rpExCzVr%?6~pOFvMrYMDfv#!AL3}smjcOWnq32Z<aHKBd(Ez;)0sOmnKI$9)rHNH-&I0errG|WbiJE1;R~0e37}{f=@X{dNKk&4?>Al!ZoY8i+RI1_t~^4Lf~7VC3$wXfz#c$66h?Pdl(GSDwI1aRA>Sfk$YoF^iK+6XU!lhe1jVOKabK-wO{ua)nTye2malT-1r~$lK-A=UXv@1BuU>mOc<DdFDQx5S%@_X7&O~K+M{zV6F24n+2JOy*dS}C@H*Pxd$(dm8l%;v&raGvOB`~hItm_e3YsRBllp*dEH3U?_<sQ4j2wVzh#8q5u@lwL9#ByIrOg!Z6?(ne*i2uKI*@MiMN%f_<RLM^3>6Aw4TH88tbY3HYrPdP6D`=k<LT3RVcj(lRIQh%UBIhkOrCxt=9d7bjCTyl<L37J_4=5Ws$?RC%vDj30Nm~vjTwuE;oFgJ}nG00cZ~2Ao=dLAbx|oxQ3JRdi)5*EYAUJS`)C=}#ad%S{(NnQWOQ%dD_Da)=r*as#>ZTNtOiRf8N%<ZS_fi%dl_q4YK*>YUvcsPG4I^%@9+YVqv@C;MpThJrI$uYo8A?2fXXuLL0t8Rjvm}w!46n9live$!r&LFN$<oPuOyZCa&}|=={0L(-cO(P2Glrx?I=|6`9}<*DIxPh%af4>ICW<^9qQBeQ=$|?q=#Y+D4^)eku#MXk_&DJ5pxVGeIFQMj5cVYrb0IFKI2(XpP#C>1w|3E?R<$(H?R)_!BrC7$s27eZfI6UqL0CkjOP>YNp0E(j0bkElF#=gbF1&LTU<{(oTiJCE=jD_6V#0=Rx}Kb-S9TsS@k2fKa$%gHET<V!nAM{?x4@{*&xHzxejnY4Y#&xz+vOTEh$0u3fZx(1@r$A0=`H<(@;6P`>xZs7uTYRg`AM@S;jjjk^0BLWDmJEJDa0aLk^!#5VOE80Xm7+mlsazq+61MNpdCc<?wDB@_UUljoX$R^bc8nr76RJAZ#}L3LI6?71~_1qKut@pIAV$LJoMS-<E$WU=&0da?K{{+t7LIFHrJP;90{RXu9b#{@xAgyTID51lFL-0hsfKtd3Y#`ru=HWcBU&#f-=CfW|M^jriohEbTP!Xg(Y;LT``y{7;%uFj!jbV)P89%`6?i#=&3;Ut-wU#E1@R>azHBk<Sj^}5;MYIBl=~Jf-CcG;b_Bh;a9uMuxvj++VUk@wB<ruCj!8ypxEkoUsEeCOuSl6;Q}Q@4ocXs5d|pDq=pI<Me>0K4xngWAp170gEB=aVxcTHVDwnd2U^AX!fFR--=;_=I083}!<htL&}mLEFw(oGa0n$u5Vu$j==`}$JSSLZn-bg)qIlBmJa-8g8ihRzY^42-{)#Pt;sZDoXsT1(<Z@w+5^(^{|7fm4XF!}$c0UU9;<<sU@^n(;?o_zSYG_B!3<~E{k(;+706$y|Vhr8kPC?9V1V>LjyiOcYj8ra2Bt%CMygAsKJaMvRmn97w=3=GxIX+0T1gXS51XJr=*7!x_Xjm7K&<EQCQ-emPuOWd;C+%F=DVd!s25ifc%iI5{!b>@e766M?g0)8wV3dnPgK?-Q*mXSQH%?;AldOxn(tP6BwQK{~Pnp5&iMy7)W-;XnL>&{IrAnhKB5aIkCB>po?7313HMdu?lfgIT!v`Uk&VgWrM34QyY9|FVK^AoJrteMl1x&FsvLDzVYGo9pYp~Y_U~%7xkW9sAHWYx2KT#?_0i__q#Wxf=mVUWdTLx1)JmEj-z5L*V-(k`n2+w89j^y0^V*Em!ymI{(of`oY(z!M7+W5h}w`ZWk8kl*4m+W2!lDYp5h3)+$m+yqeCzrj!{eQ-pd~ok?;e)dEYsmsE5a&u$fGyGGTOdW4RkUQ$2dH^^|K|`?e7G=LEVd9V@ZjD*QCRYiX6TZHVsj2wR{huf-`#_24?g(&nRf_?Fu_<vvm?UMup2XgAA7?Ew6XKx-hToENwu(S{52!J{ovmJgeLC&mEB2TAZYS{(;M&6<k7Tipfwv?f2*_kl)IUm<NN<I#aa7}Wb#w9Ae$$QW&gnkKLeFCesJ%7UPy0%IC1?&Z}-8y?+z+peS-SltgsAB?!Qwcn}wAhfAq@`Y_o8c07h^A&f5dnU&CJD+Xwgl9&iKVJ3HRX9OpTQLhr>eW-dnV5Kv`&$8cT)&>y`^b?(1AFJ`D@Y-0=~WROXQ`17AFRIDkW2dox|Ebs+Sc>>A|AkGfDiXankR3lQ0DtoS7&8p(x_5MS&Lz#%~;!6HC0L5jy>%Btl-1WXt&;=GTu@-(`RWo-NsqC(|wzlTc->v2U_=SRMN)g5x9VBqYC`ae+ddG(aqdxKYx2d_PcLgq~H-&DXUl|4&^{UW$;LSOM%iw8$?8G)5w0rVNm;;z}N;`M|-B_55|K%@#S$1v1MW@l-7K!_9a?j23-e!xd`fX>z8;{z%2Aqd!3vT`V=$8-fy*Jxkt=ZD~KX<dr8dQyQ^dbM?jqA7nJh=7JYp>qA-I>z9q>tS4QL>6nuSUed(DV@s!M2SYE|Tye-i17}rT9{@j;r#n@Eo_<y(-C(Pf>zb%C+r?m8y}b))M5h_qc2{PU#L}U6gyE?ANX2M3?ona-x&9NXeHcV9J6-D-cUX2}9U>YrAr_n}dC(rl$j3rq|MlE>)pe7YXHIPZNi3(2dV2rVXY1803O4t_<O8nJ~w{oCdo7qV~h`n|8&WB8v-N8tACQ%K~GBXNM)<Wn)*4V0B7C1|qpEbli@z5j#CN()CrM{)(cH%UQwzvwEKi&c$>OMgk#4DWw7_JoJ<$3cT`?z}xInyT-?4M9hyDc|7T0k~jQbj13{SMH9ehd+L+`$-Tj4q%c*kjZqTGWFN0?Y8NPHMT+XTffak#b41?NF_bbNQNQ(q8S5lhy0~I<j0vE0YL!ZD&%UTi8PDksB(zG2QGKiM6KN%*X?X84<9u%|3a#wGQz4R+>59*ei&|ukK%8c+&h8}HY5M17`71}oZO<5n&Y2De87o`iCGeTt2;{yokEXjq%pLA9n9vM(hah~%b^d(0I}Br`dY&Gr>O<lEYLss%#|uVEUNgLN-tv((+}<{dw>O8n+=u1}@)Gx%^cUMLd8_p85#q>OJ{~UJVcK)B1$1o0cx}&~!6oVlo>)6WtW1D)reK)?bZhkshaTIZfolE%=K+HHV(!BPuCxYAqWm3B+JE!uA?k*C_H+@B(LYN1h|@acN+(fsb<~pM7WJ5L>d}j~XLqGnXvNhtA@+_(y{28S;kXE~8r?wQXuSE_ha8rL;i(8bl<`nSQX3o}n5D}Oq~6tZm*iWy>mBp7;{FTrUN0aAY@oa@JLtNjQk`%JXF3);{42vskk|~9l7)1ow>l?jaGN=~mv*QoSA(;`ForNyX9DN+EtKvw{k1h_kG?*z(8Ii6;W{N#4@3wD01#20*TTTV2<{G5n}ei0c(4%@PsVs4Pc@4tUTP~vbhl8olCoA-tHgLm0cbx~wQ)3=_x$Ua$1Q^ZNDwa5EHA`oe<`qJ?tldcf&`X^;!#$4zGJ5rVj6)LaSa4GUqQNsZ{(;H-qNw4!?L5rgtlEib-E^O&;!sI%O+QVpeCqWf*d4)Oiu>!wX;D_FrXeq$b9!W7FVK--RoPsRD8li>OcTmXx8e~B85Kl;cD$d?ZlHfxx6wPMtNCY<aZd_3*m!sotV+^DQ(DEo;7t`r)Gqg>W3(RP+U3%3ZCOR6$)OHCZ~ejP@X#-ig-5G@c7W#y8?0PYoH?<vtEz3W}IO??lOmd6y&Y8$gq~S#aqlN%v9)lC6YO72e|{H4W*QhNwy362VrMTt&G_x)%=1!Hz=_I%0Q(eiWsRt*HOTr90p<p4su5#mcOO3b$vVM6c7CGfU67+*yg}8FUx(gGo+K<5MTMlz&bc$9W4rDQKuwHqzM8_O%aGlgR!$&K16*Tw{JX81V~y{<{B#7U9;8rzwr4_?f',
 'diagnostics_snapshot.py': 'c-qZe-H#l{aepVk|Dk<CFgw)V9*?%1VDAH<NeZT|4~ja)h%blH>}>Du=w@fu^Ko~p%K{1XVB{e{h{OmCLx^ZWP-Mr56(tD5d&t8n{y#2%Nq$w`J<~HgyLXZWClth;>5uB_s_Lrhsv2%=Y&=M%pG)D3Z0bi*S0tlR6vk2vLw_76Ssn)B$B&<gQJPFKC`kR&NTfyF6=57i#W0M=V&<oL=yP+{+t}E+eJh;Kk~9~7I!0e6Z{OnSK@vqW$ipPggdXuADdJqFT``m+zlic-806*jAUn}Bhd?n8r_xL_?_EKEzn{dioHWU&QTY-k)g%8z;h;B_xeow-+4WwQ#CoBbpHDExtn&n~<*-@g=c6Q@mJQiFE1%yFXQMCz8uTAFBS6-nN02)Nvnx*hGzPIUw}1PVz@I#w>nHk?%DhNp#-tY|{xIWNJqD7?v%DkYAR%J!IYmC&{1eA@)ns`V$XPBrZ@&E~O%o9EPyMKn{Ok%pqkd}{gGA4t&15|6jGS{`C-$du(HG|!y>Q$mw9XeT`-)a<EF6hA$(fQpc@}1Q)^V42dU&Lo(&RJ~d*$nCm`~Ky1PCtgnzoeXz+q~E1XDy@Oa~-KEW2SWBpPH2G3lt`U9oLrp%JCgJq!qEoo4SckL*8$wjrX$G=k^4%TqL$D&8!!#{RROP3LN_O@3*@5@r>*aD=?UJeOIA+Ph*D%4k@!N|6TS50l>8ph7%;{AMSXr?ibd#{fP}LmW|z0<hjg9MB(eLrGYMNvLIe3Vat^%Tqt|`K<3bArwg}>)?9)Gl4JxlI|fs0!?BcVQb5+H3wat^J6*htJ5+prn9VbLX^e2aaN?#^Rplf_g?j*OuEG8Fbm@>_v1jCf#6h@L+fBSt}7%43hrs=M=51kwyOqeBzr(;3IV58&`GU|28%jZ6s>w=ne*(9+hfs|9VOVCSvV&87`8#Hff-&_Os9T2?<56qjg!;m!zNE;oYMiajwMEmE#XATICJPBj8BrlXSLho5sn4N2m(|%pF^U*#Rm-^yQ9ql{f@&;IEj0E!r^(2@Z(|CkAs`?YI+Qt4Ckn(9MpQtTH)Lo_*s}aec=?bf8vLcKZqpobudd(^A9JN{|1v2Tz(<2L6J|cE<Oy!IP_x?TwQ$X6L>jDl4CKwy7)!7BaYRCteA_`7x~r2vp{_L?CRp@^!#g)p+TG`=`qC1Sy<EnF%*`Eo7)EYl8~nJ&?2rE4$UrSX);b_)<)1Zd@{3EF{z=uN`lu>V4_z|8Y5_tufbl-YB2XEYXKT9XK^=^L81_#4kGBGBO`y7$)TtF4C&}7w$AK8{>5f#`Fmup!z>|#?B^ZL4L^5|T(_4exgQzGM!Udz%Z%7+=uCJJvL&}z4V-Hs2$Jb6lDQmWhMKA<P_}#dB+R^O_h8svbDV;S92U{i+_JsY+2nfdS!->TI|gD^Gx5N5?2Ee!(>K`du}@D$x9S|J^*ug?7Pj6gygiR7NvEsM*1pBlt_72cjE#fJePK}?07hXPX4o7$aQLDQ@!3UtlgoXGQUQym-@D@5uDwD<TtD|BA9x$)U}o6~xD_S)6}4?hMH*!uG?52ITCnO+3p2&<i0|H|rCIQ)p`>SRy9!5!0&pdWZE%TZT@L|{=F-lm{#gf$S0b`mZOknpZ&#3+_vPp?V+BepwXbOu<$<l~CNin1N_zf0&>A!P-EXfhKF+moKfZeLX)NN&)r(K^9i<*8$>m3wLoH&6iBO~I<*#C;fWxa7{}rosdg9T+eGvfmUyJDSBSCO~5oq}FxaVj*EpKvAgb|pX29xkaDm$lNjD@q?a8$@6r*RN2el(-iVA2tLm5#t`$6OG4MIIOrMC(<{@0|K^I8y4Na@@6oHhrv@TWqIK+-o(#s%w=#muEBFhFD3QL&3t9z<~5cLq#hX3v<weB&B1M%AXWKNo}E9ZiD8Lr^pi%I;<Uoi=OJZy>XNbI?f%n9k=eA)0i3!8a{qCR$Fp4rQydy;&U83SsSfJYBNH%gQk-WGGw4VTVv?Rx#C-U<$y!gtLOv^LIaK(D_?WLdHiteou{vfcnpaS;UT>Eby0c<`Q>loi8#6ZD5(pz<6gHpI=}FBiI41;TD!ozYdlBVY^M&3w$f!~ML)x2>W3s@q)uVLbaNi&$_IkV&5MklCV4*ds@I<seiY{OvZb+z=^jP?n8ybbKlK8-dea*_>p`BR{6e=Ar$uvjYx=*rgwg-p2VwJ1J)*mL&L;l7SH8=e!D{7e1Bi^rxOmVgKM0`ntsNS9Z)>U%qXDuU#QXA&!+2PmTW)tiHx4NJ3chJ+(AqTXnbmskVo4$n4NH&sOjWY2n_ZImcJZ!{Yk{n{pxH<ZM7S{9i<nJ1q#xXN7{d@V)&o&P)<FS-)kU8nXoX{TQ;Nco+k?MEK@xRcs;y!MxlPx`(t74h>xJEV0D%2g<zcTPK|@!VP6PMcVc*AQLs#6nqs}ygZ$HxJ_$tx&bKD4LdCMtP65}qatD)K!DQbec7x~k{&?m*uRzRWPHr!D-Xj#o-)NdCIQcEK3isEz2e!sR^%J0`;v^uUWWXuW_$CwS(@^yw6tlc2=j#+mb<pM`ds99=ELu0A2bn0g`bm_j+HdSdex7KHgqD!t4hXC{@26ImxC-M-gXk$l*AeUnhib$5wln@Tf1uK>9xG+hXjQA{roG<T*?d7nBB3Iu&MTNS+UkAK7<*;&=YpYo!5OnCQrZ}*|CHojY5)_OLK}Q`($Sl8dEMtzRY4<;-*hhK&B3B>LUysZzI-vpB;@W;`IP}<lcpNv*;Im<1=~`*8U}=cE1R|M%!)qbbM!0p4UbCmqdQLHzaIZ+&w=^K%VDur<9r~^H(qisD8Z0~R90^r(8aF90L8)E38rXu^sj;n{T`e-@)^MtG`z8ymmyc@~v~=A~R<l*3S}KOh{|XyZr@yfTHMVIdYT`ap*XtAsS-9GATJzy(28buCU6CB<_S#s4BQlEGkrE;^M`Ll8I;>)AM$%}ew%ytgOPv5n=juuCSdmzfOLcUODydxma=M99x(ge+3Nixp_0dLlnQ$dA4VYveYhoQB_(zw|w_`Yu6UaU;gD*ZKPdd4L4ivSkJwxlqq41GDLc+$+Cb;RIZjoz<x4AgGd>-%(^fxgBz_rudAw>+AJQTyCew}1B%W+q7K>i?J_1E>Q>wX(I)SR&8Wv=nWTb*w$YBzSp-5L#FCL(_G-uNHFA$Ls#R^~aGFe^u(<-09QK-tV8j`!X@JzpIO3iGN-hvu|sJDi-z^dyuNZ8)?J!0xx8DDwGCavE}fm;Ocs5DBb}lW=2ymr{>{q$HF1-Uf&M(aNH@|G+*fDj2i7rMhfFI=fjuN82D7&d*1<i}tr)nVnHC5@&p~2zP)k7}vk|o!QwAC2}?=G8|7ZtoO<yr*QiD0E$MYo0N_v$?BVjozrlbPx{+;@BYyOxBh&nNA|aAfg}ot;-%qmxHI#IWYGG5j1@?THgOEbaX*ryd<S$v%Gi%|8;tP+aEM%^C^_9cP5oKlkLRZonaTyYbbdO4a@x$`(aS#GdBj3f{X8gA=(7GS2}!>%cDK~tZF-EH-N|+v@x=f9<==~Q;y_Pkfj{H$61ycX0AWr8%52|gt$ux4!_{#YJ@I`jB6e;&42!LE%d=SQAW`wJti&!pR0E9zvFPn?%^HlI?LRW!GRM!pe0KRUE5PS-Q7V%iVccXM105ue!>fxArU=S$e&cf$w}67=#64w7kP7*Eu3~u{^U%g-3uNNKAHX3V6ys7q6TY7T81HgG5kw516<I6JU;gdoFXxO)X0$#C^E@;G$1k60^w5|PN;?!p{1@b2uP%OCFn%=h0~K|d(H1@l&;&z3xcw{yR-s}kvXEas^A&<1N((xZd;50ab9HDF5^!&S7Xr8Mh?j-UZcAgRGk=Z3a53KjCaG+}C1q@R-7`5A5ff$$zZd^NzqEM-VLxPS>a+Q%LN}!Nf3pAhA#}ln9}iLhp?KBg@_AUczV)Ddd_cmc*C-ni_EOJVfM`}aTDqWUzs9<taK6&yj?Ari$vy3^zS?!$a;lE_3*q$M!@0iB9|Lte&W<21X%F&H8`C4Vp14x_Uj+~xjgpQMTDK>oxy`NLArcdGN8<xkkPr838h{p7Szq9^vEh;e!5D`rSEhp%rL;H@P`7R?Pz{mOiigy#xI2VB2%~I&S1Y{z1Du3sVD9ndufz}De)A2I;}@T*lkh3jpB|<Jw_B52W%9^XQ^L5rRYKolYgEP($d#K?#Cqg7&P$?2cR~XiWy!(%*RL-AQM_AjguuOv`*FcXa~!fa7hirBYi&ZJj$`~YYy?b=d$(_Gigz{c9Cm)UUyHG#jK>!0u6XM~m(#OoG2X3}x!vM4!!@}JwWx`)pmYIin#vzN8EWxwLd1l%qNTGqk-;%rn<ZTc<|&Z+h-i>}$|xRBgxN~vdwju!GfqD20ECIfnADd(=*Xp9z4#AOZ?ye_{9XlB+XW3($EkzK?|$ny5EI0TRjSo&S_ifiu*P8BOE~A`4SoOM;E7O&m7?ETH*w17@k0Yp2LUz2U{dfP{LTS0786eOF#oJ+GCm2@B&K_l;rf$#K0#7Z0HvQQjQ*++6POu}yyIdJQesKbEeBL_LRLac5EV>p0Z3_Qv{C4)n*^1ea4i-=s>;_gZ8L_K{|WjkQDpO+5iAlbqKz+JX<^rvA_f7pyCE9S14zi_KTU~)xRektF=>ZIUxD{(c_U58WRT>Ib+_*l<j;+$;a837<!9K^lo6ooq&-Ev9)3?jG$Ktf&4OLBIG+$*|5?*@a0&Sc)L&n~5qR<M@lK;r+2G*6KLazN21**!2}fwBZg_cVDqNstDGeA<K4mCLic#1SN~EYFV%)^|FNB4C<VB~k9T61Vpf(JcZjUq%yD*$l)Cch!^JT<S%upbB|4LuQ<I7)S_u%h@I0M9XZr@Tku|C!*GBvNs5r};KuJ$ezTS6SI)X?o)13!VLy7;>(vFT&&90J=f{4GxSM({cy+a#`9`;|E=mSJexFO+{&t7hR1*7RXdk*^{IF+#Z@<;l^VwhLKNs(v*m7MffctzPLQEM$4a{TsW){iaWR`xY#cGEzFR+d}~<>5u6#R8U8~WP1lGS9{SV*GrE1)m+>p)nlsgr2#=jg&dkR*<v<qWT$)ibY{%7rL8FR>C9MFQ4D9E^<eojs)j&3P*;JX!#wuP1IIyelR5Us9Q`?8EbP22x2u%FZ0^HdIkm68d?inP9CfJT)Th_kd|J$gBkqZiGPT^Y<<dk8|HBH2qeJ~jshQ;(ShKN?9%Us}bj@PRoF{7R8%h2ul{9cc<5x$f%JJ0q!X==(oVNGlaP}%?j7qkG6ht+Oxl&mO)fbjyA3yOPK6>@F`v;F6a-rw|$khV)9K`2FRZ2*Ho%8h3{fDnV>P;yQp**8{ZJyIC=v;zO^NWaCe;#XJ()74r6{PJf0iyElzA6w)>k!GRQczPxpsPWzgJXF-H8{Hc#&9ejrv}Fa$X7(}+#77Pa(Eg$RUJ#9Z<woVu3S1oR_REYbeSZXq~ww&%VGbO))8wHcxuR^74pm1rQaHfymGapZw?iu#fL51r`Y147~(31PFT)6`0L(x^iQMkn!g-?l_Z;u5F2i94LYw)nXRc-Y1F#ld9+dOZo$WCV{Kt&<?8C&n#l@Rm)|>gI2pWHmv&y2G_Tihu434Vvhd}~lR7Cl&I@|nhziR<0zZ>%`I;Lxw|w2G-;S0;<a$7(^{BhLs#+uI8Ib@89be&Kb+dWVPRVNbJz5jh)$j|l@O>$E_iQm-L04O!bSz*{Y#ZjPpozJ)L`Qc8uRzShrQMO;BWI=M3xRsw>q=1BaX9lJ@MISVnrSC%Em5qv?ZFXs))x#{)fTJ1Zs2oi3OP*r%2aW{?j{kc^;Ybh>lP_u3N+!$z}@olq$)%6PaO+{%985wsy3v*T3QT1I`f!f!>4dX1-0j+Y%|HivsFlzlA{6Q_}N-%4GtDZYyn-(*^0xOGRq=b>CS5wu1l(a+BG$X*%pPc%y!)MP%Krwxz~cy`q?&}ddLKHZn)~#B-+Yju?%<lY?K?-=cBE*<R<H^S;J&~k<5Y&X5mnB0$3S<B%QY`sEue@&jjGtdpK4Ers3k~p329e(s;fzBvyX&dM~OiYykjv22CM;O+fssx~a`DQGHFT_xD?uzY?Q+A78-q1=urjj^1p}9w7yinvE)~)NHjPO!clc;)-uP!Xzczh`RQHY=Tsg2@m0xxPe-mR;*cj+F7gCXRCOfa;@nVW%?YBwvkrhl-V*ZSJfGF@M^it_z-!;qyB0Xe{aLB(bge0LcFZ}T`LJqc~_!=Y<xTl$7N%d6loxdZG6`>&BoP9C5eb`UMB!=y>tJy#|MArJ$>}#&8G+I)@h`!e&=)Uzw_|%L4Bb4xT$$a*R5F2q010d1(&f<aM2R&I=KG>VJpL!YXr_Zyy~1ZSfpEVoyfTa+_X2&Ty54m!jWf%`}=%JBUU8`%bKgITfjFtGgXvugSE^o9DV+Bp$QEE0DP8>H2w`>nR1PWI28px_rd*Q6rP!~Ak+1uN8j@vJ$?G-)3;g9))Pe+YMc<I5xonGskTYZG2&%o@5E2MVUYyf`u|XrUJ^f)az>d7Wb?u?)dUGu>r$&1qyCH}Za^_EN#9@P_&8ogm$eDcdMmH3>N3}YHrJvh5y~o*ESCf|*ENglR#PdA<#!w_NK&J&Ug(Awz*fq$gZ{NO^a>_b_o{P+B7(%#cN)z*6_Npb`Jh-u!`kr2lclX$via3a3Z7IF*VHEycuaDb!ii1sCHGj>a#5RS=QOy>Jz8xF%jNoCU!)sfz&q(+#aHi3UZwiPIFs_&d}P+fwoRLBu6r}Kv7g5Io|;|lnnfuB)3Je-@mh2tIv&Wzhgw{U7ujLoYDN*GYzccXj&&`*Z@#Bi%B}j4jPzMaQ~pOMQqBOVe}JNQKOJ-Z<`eGe43RU(NB{hJx<|#Gp6A+A>0X1c($t)ev$+Y;anx5=u%JU5p%oMKc2`WoND^i}mw#w>`K@YIf?fYWj>h!#7jg8FQ;nvub6DGQozESEMZY$_QSe-rxLF0|wZi9efN45fxcXytNVUhQ70=@@hCPoM;dxHqd{;+oxqa(@0W>fo)c',
 'export_snapshot.py': 'c-oCtOK;;g5WXAme_-7MDa2NSpt}fM_)wrJ(4yD|Xto#EAQ*8Jb62mB)Fy6_|K1srlI?gmtqwNDdGec?kDAS9*H2w*JyWb`>rNGZ&VK!Q(OGa{znJD8w{6$EAK7Psj#+r<!FlGbD!?(*;0`Kicg`Gi$|?`lE}zY2(}}60Uwe=H4#thGa@)$>jqjga+e{~=ZEMykPa>8iH~2c8jEdb(S=&A`<xuhUZjM1PH6Nr8WHp^=C^0>yC55O2dC!*93B%`m=Df{aR~cWK2As&4s$9#$j$<FsE-x;x&Oe@ATwJ}wQ#zx8eNmOs?6*Jv_&~-44elTsd35<`{5G5*Z#!rNG)1dXX~lbAF0Ob+nV}u1I^{r*1>f6-ttrF2YL#|^^2u1)T1G2^I9f4NhMk;(cOu*D4_PbIh~a1fZOc|8^7&8pzOSM2H$fsaxB@+R(>5z9bz4Z89h&A!YpIfr;EP4udp-}PHx~5j*7lGLP|IPYWEcL(65)(;v_vu0QP6gd?@8j?-WGtug94HEIpeY6a#a-2dC++f@?-~39Td2Wmd|$0jpmyrvjHv}unK-(qB0`Zw0?+1h*Dvahm3u<3L8<J9acpd2kbAZ?lm@-75uvH{4R|GInq5gN`c*u!{UbAN&}T9JOt%6XS}x+{gtLsl}s-lI}XAEo(NX1TPh!|a0AE6x0YVI{;o1^3s?$N+vgS^Zq=itW+PQvyREu>_kQqu@H%E1DsA+bEe;Lkx_<r)JA^R52QP3LTMrk6W<1f$A-oo$OOVY;ZB?oIPAg{7N2~{RW`N_kCI|RpT~PO-$AR)Y>Adp2;~T06AF9p?Whx#j2BbDa-!G4`IIY!(wpk2r$P0wpMV=ZCpR!0?lt;M4K314s5LuRQ;Ypi&<UA1?CaDYnZ~ilzThtD5SZ!Rbe6byJaK#pC6~-rzYq8#Yqgu>KWE%he2MmtsQdL$65~9qY!ayj-Ch=12^^O!zi$B^$3Pb&U$dkfH(I%;Qbyq=5_m-A9aklJsl!2gbnx`YJ;@?0j5BZ}tm_o1@;djoCXFZWO_A6aNc=tFWC;pG~W5r&vLn4L2HG(WS&p)+Dgn>(SUw2N}IfKUa7Fo#^#;krpl1FkqWBl)i?;Dho&q(zZWv4*_;wU=!Lmt>F9^~sM$a3Efm}DahB^T5`80t-?xk>k!ZTm(qsO3)@$CTe@&4kPU3Po4h?{5XdLi80e_%pIp5$*%Wmw_}BWY~(*?>fL;9vYA+XBa*i2cP;*rVk1n>?NZM<do48f->YIDIfU-tietkZ5R*7dt0S66;PJXFE4u@YE&JrQgMva{Iq@th-u|%JQKp?B8v@hxAQRGaeJWFDfVVfx|@JoP)(STKf)|=$MQA(1o<K1yd1YagfBH&tHzWl4`(<Yfy34hDb4ZF8q2o2{C0nLNCaOoC1wa;UnP^04kTsZ6e;Pxl9DH+;>X8y@-L$=LTC',
 'priority_probe_snapshot.py': 'c-qZ9O_STka(7qxKOlNl0VtDL?n-)*DbXpt-F2)yJxL|4?U!0_p+FKO;UxhY0QbXNd(Jt>+)|awEr%R(dhd|KOXYv~pQO8|=L;BsyONxW%H<^nGd(jsJv}}Bg$EBFykEzwmABbWR_S!Gi5H0%Pp3&)deeNpiHo$%vynF`^2;(Qyk!#4lA`p|%zKyov`xyY?0fm9O7kpUdGC`|lBL^q_~5~V&p(?N`5J&^(``{CSryK=)wW2W?Rt|Jm3Oi%0NC3+Umag1(`}U(eJ?H}>~)n?$xH*NlB;T!PGrMuyIOJS=bv%ew7T9TrFvZaoNm;^k1zxEu#T&xddN%eWCNu}4{yy%MWTL|%Waje^y7B2DRRV>F1psgt8|?({B5xU@FAeHOk^ie_G6xAef%*?=0&_vGhVHdDWX*23f{-tRW(beFy+rb`|kMN`<HKCdt=Yv6lq?h)%5^IGD!v(kNwX-`{ww?5!8U!H=lpz!9V{6tmvllN1l&Z+gn!aRX^TrV8vtXXYUG2zPPHFu2;`~8V|!S`w#aX(yym}--0-J6=#br&<uk9c{w<I-T!&XZIZ0k<lD`n0O}dM%t~1AA{j^u!p8ep?G`8Z`DZX&FH5S+yf}}NqR5NrJk4gow1_WP2~bILHSLWA&{<N&=?Z3ARz-m2;VQpOil8T6r*khY)2yuGY?|QvzQ<6)f;$hPG$8^m!}srB9b<nII{qsXpxVsy%W@?K<9`t7!#vAiQBQy-@@?hnU@B@vm8kMz-MoV>)zJs;xMl}&b{)``W^B&$!lUQF&*F6gdp2Kf5x$sz-}4txaW*T`Y@T~_Xp+pZRQ&R)+@mHJiQjV#VwRP4Ov1p{IqdngWGnzcV?Zz-f4a&`0M74WFy3WaEnyzm*#_pG7%ke-3k@Ry%IMty3c!-;&X7dy4_c{?Rk9bz(XGUu+$_^2u<#^XBoXrOfSGjPo8+@=*@5KTz!c8qCK-7bwAkl;_yJSqI$>HR>oVxwik4Iv+99;l(f;t@vG>r!KWFx4o96j>ngC8#c|3#qU&pI5QMd_UfB_B2d+t5m|MCFHZstw=Sw7va@pur;3H1b4;|2d=<zV8ka>1KI(TlE)G-AEptm3Hw&7wqNvnyb4lD{JR)32-IdSv%r6-fd!!fGKl25ysP3o#wL<^|BeWQCP|&kvEo2iQ2g!XqZ|_x7;O-X5~ypm&<(Re<DD!s#~}mD3{KR6Uw&?M-#PN_uCopF-?<6V!UGqaJb1BSCcU9}IPUL4#iKUA)>R#P0fHVhIiOEuK+DgAKZRIVrY@MWd4PLf{X9Kb;TOX_YSGO0WlCF=uZK=ut%L+r7|2KCjYhm3;N%wyeC^Tjxk2ZKOdBHc0`LyoOcr=yZW2(Yr{Bbk2uaXJhUyt}5t8aDkt;X_3_E%bI^rZ@&Ug@1$HtB}f9$I+!QbbcqMm7BCsXj?C0H2f^qFm|R={n_MQ$J58px0eQ%~Ttcz8LGDsq;FP{OIeF_bcsS|+qe8?&kWlR+&9}I-x#Lg{#M+p^4>S*ujd<7L-_3Q<>w73-1nMm}U^u|*Dj1F6AtMhDfeMaYS-bC@_XIl;X9uDG8drXr6czrBx78B=Rt2nUlw=pa<S`(<T+t9gM2goqR2Ua_WRHidHO47gHHVKtdN!Iu;{Fmfvt*^PhRkCnaVKXiP~M7Kew2s`x0@L(PEg02vl5GGeedDJ8~<Ba>cI;n6_AEB-Q3nd$mL=X{8+2RSdx__(rT3mWvY%qU~RmJJ~br+m$b;qHJlpALvXbvX~?6);2b5B2}A80EsZ^N8K+eQaA9`=`_;0Z+Dpj=paT{daJWjd1nvs3Th|-4vr6b0)=3_@K%CKhH4CWOsSh%?f7Yk{7L~BMvTAibM*h{Kjm?dgd0AO4c)nSF383*<4%azw<~&QMk^*qx@Nykr1qZN_a#-FM`t%2Wp0C4gmLi7=hJ7!5(o-`FGdK#OxFVj3ljwWHo;ScgwGd04eBp)r-dAIV0#V7pG>@Dx7y(j2OaK;khG)?su$5VUne9#D>3Ke%(<zY5O5k1VIL(m9u#n<SwzGv2Y7FLX9OqU?v~s|6^K4d9XGOjRX&sCE-Xop^;jUaI$%bC^tYbr*BbuAiIDinQc`<WPLK))?5w4xv%4D&9kL9L5PXRVo-)8n|LgfY}ZMZSzRXL`K^*zyFG)s%IFMgN4!M5KoW5A|I7i1m^?E_fa7^THDUSvQ%>9nNt8@W5c*Z^M!f=MQ@do;_QB#Sg_r6{$&W3KoGPe>vS$0k{$L<SrR1)Q@$NMlKN+iHp&T?(J$(=&X4`N7=j$%Py^kfCeD0N>3NfM}i1@C6|am^Vy*N+;yJhEOj7e1e!y9YsR1K!RyRcU0fmFR#UokG<f9aFb*+Aj@|6fVBu#AMP`Ggfd1X)E3-{WSp#gz#AfAiZ}MhvG@CMZd;f{iiwb60#g7<{b^mqq8Y;8UB%Zhs@HJkvfnq5F=`^SDZV<OOL{-$1*M%Uu^{Sga#%QE#R7x9CP<bfCA}TuUlJVKx^MO<WeDpG1_?oo08Z>#0zxnl;mvKY1!$a>H6$UDB^y?#-q#$L-<7bj_Ir-80+Q=H)8xCOj)6Fk5%}_g8c7H!1HH4@E#Pt)gHQw<W|F}86OWjy@51_ppywN+<dLLs6tBd*pp1V}8CEel!fg|zS{S+M&qB}x!3GqEp<@utZd!SmwH$Ehg1W~^&)xKjlI~W^t>Qv!(kXMM9B<qT5Bo#DpYk8~A#k66v}4BR?)hktd{&LsnI&msEPu3bxS*j|U}8F!>V>}d=DoPWfl3vdwIEn84b*X0AqtR;j}#AMscSnOJsA$q`kvBifMz8w%Q5O=g)%ORP=ORhcfps|^o@5Z|6D|tf&zNgMo28Mz|w8-#Ar$@jr~55B(7zc8jFy)OX-9s+HU20K;p5`3Sd=3O~gR%sFv)Uoa#kPOFp^V&^aPUh4PBzm(QDA5?_C`c<#o_?t624Yo0bY7OFTsL_&gAP`Lh~4I$0iY6zZa5xyW!NlU84C%KA|Iaa@Vt(X^EigYA3S-l%ul1Vqj`xe3fzUP#+?fgFJ)R{neulI!Nj0>CawSu8R&j4&YB?C}+7?%ke0W!Gmq!-B%%CYv1xs?Get7;R8V__^$2X(wMj4>)FIu)BcxDhOoQjrQ(2AP|(YJi*#4{xbw#NVi+Nk}VB_#edcN<7x}U+`SSaHP8=8NgPA8bZL_TT&E4!Jayfd)}Tme{dru*c8_GHoB20o}~+@C6tl3;o|29P4Kd6T;*#J#wXWRQU=r!s-OY7?7vT@vk;ANfYy?L_PsT-$MLYYYjq@)vHTH&L2r|n=@nf>ps-xV2Ty)Gb`8A^c%8F}Wv6v{Jbc>s9v^&3mw2F0@40&Y-23ft=;YfDpxq2`)Vh&>6|Q&}w)AmaU;<{K^V+?`;&!fG##<o5Eikt9yobCwPIz<psNBHgI06B6EXA^}J@-*`(+#=K8x)f4a;54qR@Wf3M?<(iJQ^P0{~qE0{s#Z|RH*7LWW17`!L`Js>H(y{VzKpCORWs$R6{uHbxG=ckJqC5jgP$9x)4-3Jjzbt*;rS2A%3{Hk-jv`#*R8pGF3`_JOa9`QVndQV02L1!jIq?)pHV%1Es(5eh*3|Qn!$joX#$BPr$WFL4J`Y8B+2KosCe|q-A)o+cV>y;Lwhf<OoPA;%UX)Qr0tf49W=dA}t6-K%wo@D(EYDzGXz47+nv#B^SL4V9+GX>R1(-a!U(a#AVU-cZu<am^2sEcEeN_X>$**wx$zO=F}?t&J(gtY@w!7W|{j$1_~0~@dN*vwh8Huktkhbfd#2`4<^bFUhFP53vf||Bmw`ljaM*4EbjGr6O4WC4u!+oR^w*#RlJBKKL|t*wW>9SkJGH{ojQ~ktjJrXAkD}@w5ooZ_f31wJT0ga%9EvjsoWo;q^SX<q^;)aWIAjrR7PERHmFjnI3l5TiIm#KH}?V=2CRvyop{Ij`CH(e8`^G)G{rpU{!R0Kd^_@Ppy1ZGXK#asGc_%jP91OUVmofV)RICOue4G<%5W}Wb0iUZO4(we(uq4H&=9LXkXyi3>5DS0)Grep-Ig@8^X)2<qOF#hNC_g~15K3q7PQN#!QZKu$7?#+i18K3;n!ZV6|Ny{{skq~!PONC5zmylb(u`hH*osG;Q;G(?Xj*|E9k{wXbe&!M~sB**McagLQ%yiR~8%Dc#N>{5JN`F^#PauItFtPV1SSamhrTV<}f(RawayVw$lfoG~`J#pf%dZCwPMgqD)IP!NK7+2hk6YZ-oh~vrz8gbzGd2V+W3K&+{b)BYSK-r5T#Wt7}p`iryoDPhvzN@qo7>v4T*3dRhuX0Zg*i{Y<+SaXpt2!q+PfVD!VSLarEjNK>vE8{-SoN=mv$OT8d#(Z*&g<oJXw=tu_Z^<Rx^7e_g5d2{SJ`DWJ=!RZ1=%j_8{HGoXB<dW7!Fx9-c4o$w!yNAZ4X1jt}27y>4jSUbZR{13GA2J2qk#N|vq3j$=k)UA(*6ln6*%I-IwxiQCZT%I?XDp=AJKtk9t;=28%0sM0!Hq(90LSvu5{p4evDXCG@QSI6sLFQMqz~-UrVc(!g}4Ubdy*H45~S!Jk&HElzHMl_Kf$~_+WYq1D{q~a;Hx@aTJ%vL70dE83PVyreq&Z^%788ji<uM);Z{H9#epu{!=4b&Kk4G&Jx}sn%_`NEhwp{}sm^Z=%F#fOrGl$Yj4CFu5lS9)_w@i!f|fzU3%jobULe;9wvDwe#AQiylEb_MJqXkaO?=YA1D{Odb*+0gXoz<qx@QJ;^95O{ta)lry003>4%3~%yd@)rn3xn7YX9th=vc<(a<5E(PFl&ccIh*TsBz~etDfzp_$lb<H{M&}IR!F4m}aG7M;a5sKQ({m<#1yxE^)F-a9Ab4c1v#t{CY(W5xoB9Es?w}!Bt9`CIjKfGHTps4JAKS=@TD!X$9tV#Z6Z;WT#ZO<+^8gOigOYNLJ3C(Nmzs&`C9TedGpf(@ue-`2zJi*7hU-J9npb!;TBIcmfb{rrjR`?{ztzmYpscn&_`yspfm@zHT}<f+D0kbEtC#^Iwfa$f57z54JXx`@I*xsNoHHhXmBdKY|)jFs~5Wt{QXEuuyi=D=GQQ$KrSg$Jwdm7K>k3YfJE*s_o{!4C(((k;J|2scqj2aQE@<@7DWv?QZ*R<CgEZ$JM3!ol;osCTBSM2_uNSvoR#GS%H8#HmF_<-pmP@c$Vl%;^HcNG;H!sKpuf;K<(Mmz^`Bt`4C->d59^+wvdBtS+7kg(5dKo_)z+A39&muHSXBFRkhI!IJt}PBe&mImr}hQq(v<P!2Cg-i1u9c;~+<CEAkEIzLus|(!RfPLf>sV1PghEB=<Ub|5J;TqIQL^N5ayAB`mI>eaBvgb~TH*kM>1(w(^9pe=52I8}HPZfTjeEw`k*d(r1sA{ZLt)%JC!1S<*=W(kUf47(&a(Pr{z`r5x`+#Uk)>jo{ii#=Y8i`%$_*E``&e2ox+;^QK}4W|c9KaV;>lo};EgDCI^<KV#};tVkx(u3hw`Uj}G-aHC#e7zFV%y4MGcD~+EW7wGRrebZD}D)kHQDwHd*vt)s`AYYFOPh3VU9g+t)92}@Ar=b*_uawAwQHmh2pvJGF$%q+@y`eFMhc$oK9W&s7BQXW`I+GI+9nl*(7jp_h&1=PUx&=n1hONupVJbZF=@Q4UEAvS!PMIIbcK!ouG`i=b{Vuf{43Fs)jTYGeSLm9pYUW&;=%u|va}a))&%I%1$46AlfjaCqOBQh*eY|T-PazS{M5M}Z>jWzsJ-2-no5PV6H4^INh~ThOH74o5tPjGkMM=P}XvjR+O;qs@`2$a^z~0c<0bJ#J4pkwX8t9$_iaZWl6_NqOIY|*wyC<RoG(v$dh)5Y4MXrO&=jvVUWr>F4`I_D=KV4T3@`5_@l&7z6dO@Rq@~`3t?b~>Tek_+<jE>Ic>7uPK!7<zPr>_2FsNqOx86}@WsZN9ejFdCIbT2Kne<{#{Ph<i`ZGMsd<{H8@Nl|=VhD;974&w7FjECSC@=+^!r2s1<oCVex15#}+b%fT{L@V{~00LREq}a}V_LhNo1jOT&xU+h<T|(~|U0)4NMllIX1!-jOrH#F_$Yg1Ay7aJj*QMH;OQ|N6PeOFQuO8LCMu*C>)2T*v7(k^7$dU^?S-6Q~wJQAUbi@sKkrrT(+fX>04&)E$M(A2Z24b2u+`&y2hFYVWWGxY?VmbtgVVxrMGv~}v{1nZiAHmo__05&|jrRu6vMHT5JOR+T!P+ihatQ{^h~^{m#9*m|kqG<?)(N=kmb~7VDf?@nw+6+M+4oS59~wp@aN?4@UO!dr&>b9>YvQl{Xo<Im)uyp4l)UR8aP<z(+sj&0;2K#KkU?*@U2jV8|KSB5Au{xG1Fwbs#D!vK`N{`~3i;`5=cLIFAX+)>0)2|<ElRTj763=v!lB!{1v_<Cf(K|6m(w&A_XYG7TW!mwX>&KOLMS{7s^e%oRHJGC_iYy8{Xm$jQP$?mf;bM1L?Lh!>pL=zknkn1P5hUXaTtbchV5|b5@5UdCWOo%;0zJyt<e`mDdJ!0ry|r+X^qZIbq;4a0gywR&P89?+{}g&@DQ3Y*%HtfV{VGtQ{XYdZh}#bQHYYiNk9WPgYAmSaRrzm-F*dl^-kMP<i<H<D-s%J;^tX$CphZpIx`cCK-go%IxDG2)9wT>gtq{jD!gfo@zyCM$YCox-YWL~{=J9;gi&s@ZAnh_tV+UYG37=eT~<7Y&Cc-rQoe>E#T=AK<bf0gS<j?>5&blyTMj9qP_E6&cn&K9`8Yt2;yV&b*ohbz)cq}nO~Bf}9<ykesbe;?ImOOfFio|Zl@DOokjZz#1ch5#AO@lEU9&z1R<milrEnekQB0?6He7&LatK?l0vA+tyJBM~Fc<Hyo4gbJB$kj|BBH^<d_fd}EA%2#o_~`Z&guds-voSP==8v1;jRS#X_q-D{V9$(;Yk!{C7_K$9JPN9rVNr~9r3maly->qE1<!F>x;xU$~ExFw@#oe2n^PFTp+4yMbZ27w7_Wnc)E<3;eoInaSQ+vG7Qt-aaqsKAyzr_e)sOp_wSG2MJLDaPoh7(IXbrP$v7BXBpgR`eWb)YL7f|3E(@{S;kY4&QD}lIZ{<QFC%lKWtb|(XM8ssfZI9C4>^i%fHc7g$xp_eQrgvKU@5ut(!=WWrF|phO{f<BiORmIb8Cv64tAnde8Xp%YrTw6)?jRkywNahs<pLMZDU&J>GpwO^06~|@WO&a9Yd{Au>e6sXho1%d+@&VhKgxxpD_p&{l%We6?WI&vodu;3KG4%xM0|5!e#z-M1PfQpU$?W#4?T%+73J7QU5RJ75`dtC8_&@)KnSNkzXGB(es`yN`F(QDztC5|CC5vhRzzwDOmQGO5!50|>w&pQCT6X50vv0`{wc3Tcu^=31IE4t?hWGb;dB&&yW;^N+8jwFk9NEE@R2>XV5#I#&5@ebXsKV$1l>ylA@Wq&*#ZelyZ_Ms?VS$Ku$8W;XboRA30qKwq-|MXQ#Xogs_N-r|4d8iJ*oW>#HAEJdMn6N0L}`K>XMp};}l9PLB1|3)b*AvuKJElRyEh`-*FZ}+eKS-=eDyom?TOkU?NKf1bBoMz{(bn+}Xw?GU~x?W@oZa!zaIWP(ql9CxGJ7WEE%UzSM!&q$qX!8MPh;S$u&(Vi?4D85fv~!(l6}O-4778a%B~alkuk#|ufcigDuPPl~_%d$#mu+v{Kc{EwA)zWmF7f^K8=%b))3+WYT+`k(*!<<I{*^|D|7{Ld-Wfj5)C{7VW!7Qg)IzpAj7vkcr&XQ%}roHhxRl%aSh16ybTtUz%ikyi%PB~%JxJYLs8NFv2CMNK038(OcC;>006!(AQ(E~Auutp3)14pYduRGdIXZv+Q`ia0PJc*>uF13sj*+E?W$*wvGu838Z>HMB|B&S=G2)mqXimooy%x_3%)MPtX>BL9+ioo_uzUE(db;Mc2a*zAUER9K364)&i6o1q`g#bX_?R<el2s_9rBq6JSU9qfB%x|l{h97UIwGK(lMFYsPBBB534PytLi<+~bRCKINfa=XD+;Wm}+zfUGd^zM5YjN*!&2@Tv?S7f#n*{k1`h=u1W4Jpq2AT@hTNMatv6|^`S$T=pJ*i9y~1M{KThM14Z!kkqA;v$fght@x{T2o*#V7CpKC6oFhx-%7YF*PIYrwC7W4yDdKE<wTt^%Nx5gF9pQWGp%fBz>C{+=H}EpdJXC9Buc6TV^VFrbw#<F(j!B@3e#28=)Z_ZpoA8@FOAcTF0BLK@X;@j}&?uTz5L#MAP*QEFdh`eT`L5z`I8XRi{FFqJlGiz#r+%K6k|QOiAO65@$K&_D=HCC!Icz&kV6%_sQP142{-?eS^V^3hpQ%#Z~jcQ-7D+k=ouL>~{1i9b)XI|5P#8=%}S8t@H@WhpWZqD!yTEb{lT8g)w2C95(G?t~oq-*XN+93&cZ7>cYMVTZ!`Xy8ahrB+1&q8^#z&fLq=G-fP)Lb@wyk7Vu=C=WbqMI%V=RkM!H%4I{|LNr)ff1m^H|+7Z>KW+_$o9QZnJ6aMXRI!BTw2~o($6i<%7x}3XjZr#q?I|ER~(%bsEu~yA>+9Avk(+47IwDlrmM4dipT3%d=HmM7f*s!&IXQTFr>{?BEy>oT)k)ef3a{qr6S7;>5=?(-GK67sMd+lCyHyyO0DVAAi+x2Fh_ovpYV%uG*>D{z5XvWG~(P;kU&Y?Gu+``cQa^^dG%d2yr9O(>Y7eq%ufS|{A3z^xUweueOdBO!n=z#aY+Eu`_%+&6*a_A9Ym_2EyWcadeuafJ7HJYm<Td2E-u6DQhC2S{ymvr0q&?QMUoI~s!Se`KriNwILlJq5UD<*qRd9iaP01E#=4TX#Q<UTF$!o41?osK6Fb`i*ZUI_HLkapw8lyaYpa*xQ%=ORrm;o+cfnl-%rTbj7WK-~_42}T_(mZo7ebJ|#-@c=K2P3X2gL`^WAo5&)sH=!j_SkH?Z2Dm;(C)h8TTe!45-@SMf9ew-eP$cA0moWxquov0w4j5MYOTod+#qbZm`SwM0_|1!d@NGjttJ)sbMJlDGa18tXH*a45U8H*Lxx1Bn9E3uYoFbw7LL)bFF3TI(*yAtJRTHe1Y;*DssTBx7l*AOiMNz<)Jz%flsRN&Jn8cV{{}pIduT;8H53}KxR!nr$=N1Nq7i#qMYjNlzuK@7Mq4TAlix7Pi#7GnLHJ_fNLL6g<&u%T_p9#>im#lEewPOKCq$MNFew&G?A2q_e#7(HE27zelk0ro29Th}5C(KvqRbSvB?+GZ6tMkErI1I}?9Kt*UE`y@Mu=?#|$}Oc!d~=`5=93}?50$||6AHgNXaF-`G+<_B`3N9MIQR%Gs?YB>#bfIwJmXs%#A33>vFyug@Vu{JT%Au?kz)>5l@6QW{)l%V^kGdbs8_031OMYJMq7<tPn7LD0MFAa(7Z$CN2)Rv=^6hKvo#l3<v>-TqJt*5xI}Ttsw;{P4D5@72G*zO7-F`>0;3&0Hd<}K*$~nLrA6MLBKy_AK<lk}*hnL^qEXBGF4={)SV^mus85kVs%hgR39igJ;>$eY;nAyC0RyQAEebi9Ggdl4@TjCn-Gst>kn73m>d81JZL+UZWttbcSA||e0&7k!RcNdYHpY~zQ?PbYj2?X~D1==8Q(#ocWznA&GrKfiP!r#s6A9i+(*jj%^@DsT7!@1b|6ntIHS=d4k7Q4E3_dLJ1&Dhn4uWGd#1A43if>o&JbMjaq6GF$jZCCN8=2ISZ8{6chBtJKxZKY228yuIcjwxqGXP$9rrxy?z$c&!2RaEtky98cB~k*Bf40K!USUVhAVWQCtO7V1WcBoF@J$2Qf!SJQ&mvth;Dby1-qG>zUcPu8ef{pu>yzWxM`K80ouQL{AzUZ{@2?J{7q4E8{X<`gP-g5gdc5(889-zs-9t*{z36n3l`ataYgOns7w}}h@C-0zHN)^9qvqR}Z;z=Q?m}ky`;((L-=5qf4K7NPQsslXH&~akFV!e?(IO3i_-e~N-~wvL2GaZW#JY}U6h4G}%(FN(OOUm|$nd)fpmGF>m*RB&*l!M)JOrUq3pK(UAx{MA6EIC=_i&p*lndskVrK_|eHOU+wR<~o1HtxCk$I1OVQmKF=R{0&7v@W!-~rB2fh8JlSfjHBZJKm>N^H;<+681KgP(pv4Hblt)!7J3d=5rZJ9FIpR#s#<*5P!@t<PF;qgY;ZYn&2W#25w2fbREY2WPEY<lgz9Teno_PPunCjh)rLVIB?J;;KRXAtMy|41lRAE`VJZ{-G=%{ZKv(et7>y55Oz5+Jw^@FRklqY9`+!5z(Q#ww^+AtPS>sj|f~SLBHX5gjP#BlM)0!cjSR}q{wUu9O&9>uX8%SC3$L+L!!k^<`#&IZBgc^RwE}wSz14f9b1ndY44E>-(ZmC11=rl&2kNEPvMpiCD^mRp%kfKrW}{FGjg;M56#<`6sSZp+_NqjOz|{lOJ{zP4vvP$yKva;kZM}`vW+awYb4qA{JSe}h1J<s-m0(V-9>BL`=l3D{h@@916I57>M-R59a*Jn*A7#DXhvIbWG^4R6R@2W*DBgqTdc<zfZP@Edl!|6!!<y90VofFZubEJJ)}=@P1V;s--|~c9lIxtHOx>@<s0!v(mGcR!BaDmqCfz<69W!W_Jqv$m;R2r@R3BM>A@_y7?jKPU`w8VI~s}aD?*#eb^$I6i@_${>}(_~<pa#VG2TCTI_!Tr92%ZcdO;1UZ-a%E2V(VcFE%o~qqmu%<RQim{XaVJ?*JT_TM1{wB)>9-XOWi!h<FjnRNm}NV702@M>|+RO`!D!8khY9O68|4WleO)n8urUl9G+jU*sSyf?SBGNA){7PFhxNU{N6VtIyWxOT(%E{x~{(arn(~^!CNcH_Gb)_W>JLapQ)PawEP>>Kr<5kS8zya18Q>;A|k(`SOc5g3QN~!oi7TftH00N<GT-%5pQ)Itk3J!pR0D>FE3tKJG(20<A7WTRbu)ufuo8FHVl5*KeZlU%ozi^ZjjuCdP^;ga%bPbc$g(wqL;08S`KHt>SXbf1OS^AK_Ee@R^WKuSrHpb!%RzyBV<|s!3*iV<TEs)uGmsb4d!sOO5I1_#M}*ubCYq-R*qGCbS$hxzI>_|EK{^P)XN_xCW;J^G)nx@R9Lo;;oJ)SgI{Mn##!FZ3cJ$bE&{^co^_J%p;&9?J`dQ^afU76(6b%$41!Wt8}rf>c`+iNq1UR|DABtlNx$bZp)7Vl`^svae&Ht^gdU{IMiAgW;nSC^jCIVWKDh!?vq*_ujWAAZe!$dtG^}9AJ8O712y^Qd!qO``uKds6C@$(`5qPdPKOVXryCD|8a)5*!k}s+0<~tTFexHEg*(J5GykpMSWvYIbqpoTh3=0n-PzXv;UlT-IXy1@WoBW?hHL`6oCL%NeL5y`pP@E(W!GRGjXwRz??f>IZO>3Of+1O+OC}cg(b^dZlqIA{^)aR9SBVsvyZt65f3b7_xMgZH_0I<h*S0I;cppm~iGvR6NqMYr$qXeJ_>eLP8~C9UPl~dNp^5MFJoAJMQIaomiGn*`>j_<op>gom4x1x>o()MsW$vkEPEqN3AazP-P?`)!Xyo&0rchB#!$`nTHo{a@tYZB(Wv5<i4^lN#;5L!~hy&|WG5cBs{CHPxC|f0DQJ3{pBQuA%w3`=7U$&e76aH`?am>NelG>-a?zqES!?F_*p>^L-`L?f3cBPfL&zJm`DSnFlx<ys%`rP8K&n)U#%X>u>w363-w4)Z|bA}|;-1Hr&P#<1td~S+8KsaYOeNRdU*~)NWPNGh+hNb(fP|CH$;WKA&I=mR|_*hYkKGXOtFMhz7Q$CsBG46GK!J@Wu<(N^UuqdLR#^^G{uet&LI=+baP@;W6lFa)h@^H0wq-Kx<qSPIBUL2k0qn_Et%Is115WaJAk2dE+GrvbmklVddeAEa~q+x55n`+$ie*ur@5Jm'}
EMBEDDED_SHA256 = {'audit_snapshot.py': '4d08e3ea1613e4248d35ebfb40716a5fafbf1a6916bfe35a99a74686d8093bc7',
 'corpus_runner_snapshot.py': '878ea8299597087c7201ba3d4ac4b9cd1a61958031452633b71dc1dc020c2039',
 'crawler_snapshot.py': '80bf08dd7a10b59ca44b02353f1cd6add14da26e7a497a011d34681dc367d5e1',
 'diagnostics_snapshot.py': 'c82968c6617017aba772a755218df07e84a7ef2b550f0c588b2eac3933ed159e',
 'export_snapshot.py': 'ac461b827b046ed7ea8fc4488a85d62392e4a95920ca9be7e8d714d3737a4023',
 'priority_probe_snapshot.py': '079dba5bbc45f2d22dab939ea51ac7eb4db893425d7a0aaaed2e6cfc380430d3'}


RUN_DIR.mkdir(parents=True, exist_ok=True)
for name, encoded in EMBEDDED_FILES.items():
    target = RUN_DIR / name
    if not target.exists():
        body = zlib.decompress(base64.b85decode(encoded))
        if hashlib.sha256(body).hexdigest() != EMBEDDED_SHA256[name]:
            raise ValueError(f'Snapshot bị hỏng: {name}')
        target.write_bytes(body)
CRAWLER = RUN_DIR / 'crawler_snapshot.py'
CORPUS_RUNNER = RUN_DIR / 'corpus_runner_snapshot.py'
AUDITOR = RUN_DIR / 'audit_snapshot.py'
EXPORTER = RUN_DIR / 'export_snapshot.py'
# Diagnostics chỉ đọc output, có thể cập nhật riêng mà không đổi crawler/hash resume.
diagnostic_body = zlib.decompress(base64.b85decode(EMBEDDED_FILES['diagnostics_snapshot.py']))
if hashlib.sha256(diagnostic_body).hexdigest() != EMBEDDED_SHA256['diagnostics_snapshot.py']:
    raise ValueError('Snapshot diagnostics bị hỏng.')
DIAGNOSTICS = RUN_DIR / 'diagnostics_snapshot.py'
DIAGNOSTICS.write_bytes(diagnostic_body)
diagnostics_archive = WORKDIR / f'{RUN_DIR.name}_diagnostics.zip'
from contextlib import contextmanager
import traceback
LOG_DIR = RUN_DIR / 'logs'
LOG_DIR.mkdir(exist_ok=True)

def refresh_diagnostics(strict=False):
    try:
        result = subprocess.run(
            [sys.executable, '-u', str(DIAGNOSTICS), '--out', str(RUN_DIR), '--archive', str(diagnostics_archive)],
            check=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace',
        )
        with (LOG_DIR / 'diagnostics.log').open('a', encoding='utf-8') as stream:
            stream.write(result.stdout)
    except (OSError, subprocess.CalledProcessError) as exc:
        with (LOG_DIR / 'diagnostics.log').open('a', encoding='utf-8') as stream:
            stream.write(str(exc) + '\n' + str(getattr(exc, 'output', '') or '') + '\n')
        if strict:
            raise
        print(f'Tạo ZIP chẩn đoán thất bại: {exc}', flush=True)

@contextmanager
def diagnostic_step(name):
    try:
        yield
    except BaseException:
        with (LOG_DIR / 'notebook_errors.log').open('a', encoding='utf-8') as stream:
            stream.write(f'Bước: {name}\n' + traceback.format_exc() + '\n')
        raise
    finally:
        refresh_diagnostics()
if (RUN_DIR / 'crawl_config.json').exists():
    saved_config = json.loads((RUN_DIR / 'crawl_config.json').read_text(encoding='utf-8'))
    if hashlib.sha256(CRAWLER.read_bytes()).hexdigest() != saved_config['script_sha256']:
        raise ValueError('Snapshot không khớp hash của run cũ; không được ép resume.')
if not requirements_path.exists():
    requirements_path.write_text(''.join(f'{k}=={v}\n' for k, v in versions.items()), encoding='utf-8')
print('Đã chuẩn bị snapshot và ghi phiên bản dependencies.')
refresh_diagnostics()


## 4. Tải hoặc đọc danh sách URL

Nguồn được cố định theo revision và kiểm tra SHA-256. Chỉ đọc corpus URL;
không dùng query làm nội dung hay nhãn. File Input có thể ở vị trí khác nhưng
được copy vào đường dẫn ổn định trong `/kaggle/working` để dùng lại khi resume.


In [ ]:
with diagnostic_step('source'):
    import requests

    def sha256_file(path):
        digest = hashlib.sha256()
        with Path(path).open('rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(block)
        return digest.hexdigest()

    SOURCE_PATH.parent.mkdir(parents=True, exist_ok=True)
    if SOURCE_PARQUET:
        supplied = Path(SOURCE_PARQUET)
        if not supplied.is_file():
            raise FileNotFoundError(supplied)
        if supplied.resolve() != SOURCE_PATH.resolve():
            shutil.copyfile(supplied, SOURCE_PATH)
    elif not SOURCE_PATH.exists():
        temporary = SOURCE_PATH.with_suffix('.download')
        try:
            with requests.get(SOURCE_URL, stream=True, timeout=(15, 90)) as response:
                response.raise_for_status()
                with temporary.open('wb') as stream:
                    for block in response.iter_content(1024 * 1024):
                        if block:
                            stream.write(block)
            if sha256_file(temporary) != SOURCE_SHA256:
                raise ValueError('Hash file tải về không khớp nguồn đã chốt.')
            temporary.replace(SOURCE_PATH)
        finally:
            temporary.unlink(missing_ok=True)
    if sha256_file(SOURCE_PATH) != SOURCE_SHA256:
        raise ValueError('Parquet không khớp nguồn đã chốt. Kiểm tra file Input.')
    print(f'Nguồn hợp lệ: {SOURCE_PATH} ({SOURCE_PATH.stat().st_size:,} byte)')


## 5. Chọn ID và tạo manifest

Nếu khoảng chọn có ID không tồn tại trong nguồn, notebook báo số lượng thiếu,
rồi lấy đúng các hàng thực sự có trong khoảng; không tạo ID hoặc URL thay thế.
Bước chuẩn bị hiện quét toàn file để kiểm tra nguồn và thống kê hostname, nên có thể mất một lúc.


In [ ]:
with diagnostic_step('prepare'):
    import pyarrow.parquet as pq
    from collections import Counter
    from urllib.parse import urlsplit
    from IPython.display import display
    from datetime import datetime, timezone
    import time
    import uuid

    selected = pq.read_table(SOURCE_PATH, filters=[('id', '>=', ID_FROM), ('id', '<=', ID_TO)]).to_pylist()
    selected.sort(key=lambda row: row['id'])
    if not selected:
        raise ValueError('Không có URL nào trong khoảng ID đã chọn.')
    if len({row['id'] for row in selected}) != len(selected):
        raise ValueError('Nguồn có ID trùng trong khoảng chọn.')
    missing_count = ID_TO - ID_FROM + 1 - len(selected)
    print(f'Chọn {len(selected):,} URL; có {missing_count:,} ID không tồn tại trong khoảng.')
    print('Phân bố hostname:', dict(Counter(urlsplit(row['url']).hostname for row in selected)))
    display(selected[:5])

    LOG_DIR = RUN_DIR / 'logs'
    LOG_DIR.mkdir(exist_ok=True)

    def log_event(name, event):
        with (LOG_DIR / name).open('a', encoding='utf-8') as stream:
            stream.write(json.dumps(event, ensure_ascii=False) + '\n')

    def run_script(script, *arguments):
        # Output hiển thị trực tiếp để theo dõi tiến độ; không dùng shell.
        invocation_id = uuid.uuid4().hex
        started = time.monotonic()
        operation = str(arguments[0]) if script in (CRAWLER, CORPUS_RUNNER) else Path(script).stem
        log_event('invocations.jsonl', {
            'event': 'start', 'invocation_id': invocation_id, 'operation': operation,
            'at': datetime.now(timezone.utc).isoformat(), 'arguments': list(map(str, arguments)),
        })
        process = subprocess.Popen(
            [sys.executable, '-u', str(script), *map(str, arguments)],
            cwd=str(WORKDIR), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace', bufsize=1,
        )
        returncode = None
        try:
            with (LOG_DIR / (Path(script).stem + '.log')).open('a', encoding='utf-8') as log:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                    log.flush()
                    if operation == 'crawl':
                        try:
                            progress = json.loads(line)
                        except ValueError:
                            progress = {}
                        if isinstance(progress, dict) and 'completed_this_invocation' in progress:
                            log_event('progress.jsonl', {
                                **progress, 'invocation_id': invocation_id,
                                'at': datetime.now(timezone.utc).isoformat(),
                                'elapsed_seconds': time.monotonic() - started,
                            })
                            refresh_diagnostics()
            returncode = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
        finally:
            process.stdout.close()
            log_event('invocations.jsonl', {
                'event': 'end', 'invocation_id': invocation_id, 'operation': operation,
                'at': datetime.now(timezone.utc).isoformat(),
                'elapsed_seconds': time.monotonic() - started,
                'returncode': process.returncode,
            })
            refresh_diagnostics()
        if returncode:
            raise subprocess.CalledProcessError(returncode, process.args)

    manifest_path = RUN_DIR / 'manifest.json'
    if not manifest_path.exists():
        run_script(CRAWLER, 'prepare', '--source', SOURCE_PATH, '--out', RUN_DIR,
                   '--size', len(selected), '--id-start', ID_FROM, '--id-stop', ID_TO + 1)
    else:
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        if manifest['source_sha256'] != SOURCE_SHA256 or manifest['selected'] != len(selected):
            raise ValueError('Manifest không khớp nguồn hoặc số URL đã chọn.')
        if Path(manifest['source']).resolve() != SOURCE_PATH.resolve():
            raise ValueError('Đường dẫn nguồn của run cũ khác notebook này; dùng run mới.')
        sample_bytes = (RUN_DIR / 'sample.jsonl').read_bytes()
        if hashlib.sha256(sample_bytes).hexdigest() != manifest['sample_sha256']:
            raise ValueError('Hash sample của run cũ đã thay đổi.')
        with (RUN_DIR / 'sample.jsonl').open(encoding='utf-8') as stream:
            sample = [json.loads(line) for line in stream if line.strip()]
        if {r['id']: r['url'] for r in sample} != {r['id']: r['url'] for r in selected} or len(sample) != len(selected):
            raise ValueError('ID/URL của run cũ không khớp khoảng chọn.')
        print('Manifest hợp lệ; giữ checkpoint để resume.')


## 6. Crawl — chạy lại ô này để tiếp tục

Tiến độ in mỗi 25 URL hoàn tất hoặc khi hết lượt. Chế độ robots theo `RESPECT_ROBOTS`,
giới hạn một request cùng registered domain, có retry và checkpoint theo ID.
`WORKERS = 4` không có nghĩa bốn request đồng thời đến cùng website.
ID đã có checkpoint, kể cả thất bại, sẽ được bỏ qua khi resume.
`MAX_NEW` giới hạn URL Requests mới; Selenium tiếp tục các URL lỗi, text rỗng hoặc
QC lỗi đã có checkpoint nhưng chưa thử browser. Chrome được kiểm tra trước lượt Requests;
nếu không khởi động được, ô crawl dừng và ghi lỗi riêng, giữ checkpoint hiện có.
Kết quả cuối ưu tiên engine có QC tốt hơn, giữ Requests khi hai kết quả tương đương.
Parser 120ask giữ câu hỏi/câu trả lời; cnkang giữ thân bài, bỏ menu và bài liên quan.
`records/` là kết quả cuối dùng để audit/export; `requests_records/` và
`browser_records/` giữ bằng chứng của từng engine. Chỉ nội dung qua QC mới được xuất corpus.
ZIP `<run>_diagnostics.zip` được cập nhật sau mỗi log tiến độ và khi lệnh kết thúc,
kể cả ngắt ô crawl. File có một số mẫu record/raw để debug, không nén lại toàn bộ raw.


In [ ]:
with diagnostic_step('crawl'):
    corpus_settings = {
        'user_agent': USER_AGENT, 'delay': DELAY, 'attempts': ATTEMPTS,
        'timeout': [PAGE_CONNECT_TIMEOUT, PAGE_READ_TIMEOUT], 'budget': PAGE_REQUEST_BUDGET,
        'respect_robots': RESPECT_ROBOTS,
        'robots_options': {'robots_attempts': ROBOTS_ATTEMPTS,
                           'robots_connect_timeout': ROBOTS_CONNECT_TIMEOUT,
                           'robots_read_timeout': ROBOTS_READ_TIMEOUT,
                           'robots_retry_budget': ROBOTS_RETRY_BUDGET,
                           'robots_failure_ttl': ROBOTS_FAILURE_TTL},
        'browser_mode': BROWSER_TEST_MODE, 'browser_user_agent': BROWSER_USER_AGENT,
        'chrome_binary': str(CHROME_BINARY) if CHROME_BINARY else None,
        'browser_load_timeout': BROWSER_LOAD_TIMEOUT, 'browser_wait_seconds': BROWSER_WAIT_SECONDS,
        'browser_attempts': BROWSER_ATTEMPTS,
    }
    settings_path = RUN_DIR / 'corpus_settings.json'
    settings_path.write_text(json.dumps(corpus_settings, ensure_ascii=False, indent=2), encoding='utf-8')
    arguments = ['crawl', '--out', RUN_DIR, '--workers', WORKERS, '--config', settings_path]
    if RECOVERY_RUN_PATH:
        arguments += ['--recovery-source', RECOVERY_RUN_PATH]
    if MAX_NEW is not None:
        arguments += ['--max-new', MAX_NEW]
    print('ZIP chẩn đoán tự cập nhật:', diagnostics_archive)
    run_script(CORPUS_RUNNER, *arguments)


## 7. Báo cáo, kiểm tra và xuất corpus

Corpus accepted chỉ gồm tài liệu qua heuristic; chưa chứng minh chất lượng bằng chấm thủ công.
Audit chỉ chạy khi đủ checkpoint. Nếu đang crawl thử, vẫn xuất phần accepted hiện có;
manifest notebook ghi rõ đây là corpus chưa hoàn tất.


In [ ]:
with diagnostic_step('report_audit_export'):
    run_script(CRAWLER, 'report', '--out', RUN_DIR)
    summary = json.loads((RUN_DIR / 'quality_report.json').read_text(encoding='utf-8'))
    display({k: summary[k] for k in ['selected', 'completed', 'pending', 'accepted', 'review', 'rejected', 'status_counts']})
    if (RUN_DIR / 'access_summary.json').exists():
        display(json.loads((RUN_DIR / 'access_summary.json').read_text(encoding='utf-8')))
        print('Lỗi theo tổng URL dataset giảm dần:', RUN_DIR / 'WEB_ERRORS.md')
        print('Đối chiếu Requests/Selenium:', RUN_DIR / 'requests_vs_selenium.csv')
    if summary['pending'] == 0:
        run_script(AUDITOR, '--out', RUN_DIR)
    else:
        print('Run chưa hoàn tất; bỏ qua audit toàn mẫu. Chạy lại ô crawl để tiếp tục.')
    if summary['accepted'] > 0:
        run_script(EXPORTER, '--out', RUN_DIR)
        corpus = pq.read_table(RUN_DIR / 'corpus.parquet')
        print(corpus.schema)
        display(corpus.select(['id', 'url', 'title']).slice(0, 5).to_pylist())
    else:
        print('Chưa có tài liệu accepted; xem quarantine.jsonl và QUALITY_REPORT.md.')


## 8. ZIP chẩn đoán — tích hợp trong Output

Chạy ô này để tạo lại gói debug từ checkpoint hiện có, kể cả khi crawl chưa xong.
Không cần file Python riêng, không tải lại URL. ZIP chứa:
- `overview.html`, `summary.json`: tiến độ, tốc độ/ETA, trạng thái HTTP và QC.
- CSV từng ID, lỗi/pending, request/retry và redirect.
- Log pip/crawl/Chrome, kết quả preflight, traceback notebook, manifest/cấu hình và phiên bản môi trường.
- Snapshot code, bằng chứng robots và tối đa 10 mẫu record/raw để kiểm tra parser.

Raw mẫu có ngân sách 10 MiB; các mẫu bị bỏ qua được ghi trong `sample_manifest.json`.
Giải nén ZIP và mở `overview.html`. Có thể gửi nguyên ZIP này để chẩn đoán lỗi.


In [ ]:
from IPython.display import FileLink
import zipfile

refresh_diagnostics(strict=True)
with zipfile.ZipFile(diagnostics_archive) as diagnostic_bundle:
    diagnostic_summary = json.loads(diagnostic_bundle.read('summary.json'))
display({key: diagnostic_summary[key] for key in ['selected', 'completed', 'pending', 'quality_status_counts', 'timing']})
print('ZIP chẩn đoán trong Output:', diagnostics_archive)
display(FileLink(str(diagnostics_archive.relative_to(WORKDIR))))


## 9. Đóng gói corpus đầy đủ để tải về

ZIP chứa cả raw, records, snapshot code và corpus để truy vết/resume; có thể khá lớn.
ZIP riêng `<run>_diagnostics.zip` nhỏ hơn: giải nén và mở `overview.html` để xem
tiến độ, tốc độ/ETA, HTTP status, lỗi và cờ chất lượng. Có CSV từng ID và log cài dependencies.
Tải file trong Output hoặc qua link bên dưới. Khi đổi session, cần giữ output hoặc
thêm run đã giải nén làm Kaggle Input và đặt `RESUME_RUN_PATH` trước khi chạy lại.
Resume không tự thử lại ID lỗi đã có checkpoint. Dùng `RECOVERY_RUN_PATH`
cùng `RUN_TAG` mới để thử lại lỗi bằng code/cấu hình mới và tái sử dụng raw thành công.


In [ ]:
from IPython.display import FileLink

notebook_manifest = {
    'id_from_inclusive': ID_FROM, 'id_to_inclusive': ID_TO,
    'dataset_revision': DATASET_REVISION, 'source_sha256': SOURCE_SHA256,
    'selected': summary['selected'], 'completed': summary['completed'],
    'pending': summary['pending'], 'accepted': summary['accepted'],
    'complete': summary['pending'] == 0,
    'packages': versions,
    'exported_files': [name for name in ['corpus.parquet', 'corpus.jsonl'] if (RUN_DIR / name).exists()],
}
(RUN_DIR / 'notebook_manifest.json').write_text(json.dumps(notebook_manifest, ensure_ascii=False, indent=2), encoding='utf-8')
refresh_diagnostics(strict=True)
archive = shutil.make_archive(str(WORKDIR / RUN_DIR.name), 'zip', root_dir=RUN_DIR.parent, base_dir=RUN_DIR.name)
print('ZIP:', archive)
print('Corpus:', RUN_DIR / 'corpus.parquet')
display(FileLink(str(Path(archive).relative_to(WORKDIR))))
print('ZIP chẩn đoán:', diagnostics_archive)
display(FileLink(str(diagnostics_archive.relative_to(WORKDIR))))
if (RUN_DIR / 'corpus.parquet').exists():
    display(FileLink(str((RUN_DIR / 'corpus.parquet').relative_to(WORKDIR))))
    display(FileLink(str((RUN_DIR / 'corpus.jsonl').relative_to(WORKDIR))))


## 10. Tự upload output lên Hugging Face Dataset

Repo đích: **AncientWings/AncientWings_R2AI_STAGE3_ds**. Bật Internet và gắn Kaggle Secret
**HF_TOKEN** có quyền ghi vào repo. Run All sẽ tự chạy ô này sau khi xuất corpus/đóng ZIP.
Không cần đăng nhập qua widget hoặc tải output về máy.

Mỗi run được phân biệt bằng `RUN_DIR.name`: mặc định `ids_<ID_FROM>_<ID_TO>_synced`.
Recovery có tên riêng; nếu crawl lại cùng khoảng ID với code/cấu hình khác, đặt **RUN_TAG mới**
ở mục 1. Chạy lại ô upload cùng run sẽ cập nhật đúng các đường dẫn của run đó.

```text
corpus/<run>.parquet                       # Đầu vào RAG: chỉ tài liệu accepted
corpus_jsonl/<run>.jsonl                   # Cùng corpus, định dạng JSONL
archives/<run>.zip                         # Full raw/records/checkpoint để resume
diagnostics/<run>_diagnostics.zip           # Gói chẩn đoán
manifests/<run>/notebook_manifest.json      # Khoảng ID và số lượng bản ghi
manifests/<run>/upload_manifest.json        # Danh sách file, kích thước và SHA-256
```

Mặc định upload tất cả các mục trên; có thể tắt JSONL hoặc từng ZIP trong ô cấu hình dưới.
Đặt `HF_UPLOAD_ENABLED = False` nếu chỉ muốn giữ output Kaggle.
Run còn pending (ví dụ thử `MAX_NEW = 10`) sẽ **chưa upload**; chạy tiếp mục 6 rồi chạy lại mục 7–10.
Run hoàn tất nhưng không có accepted vẫn upload manifest/ZIP; không xuất corpus cũ còn sót lại.
Upload đọc file từ đĩa, không nạp cả ZIP vào RAM. Nếu mạng lỗi, ô thử lại tối đa 3 lần;
có thể chạy lại riêng mục 10 mà không crawl lại. Kết quả commit được lưu tại
`hf_upload_result.json` trong run. Repo đích phải tồn tại; token/repo lỗi sẽ được báo ở ô này.

Tài liệu: [Hugging Face upload](https://huggingface.co/docs/huggingface_hub/en/guides/upload)
và [Kaggle Secrets](https://github.com/Kaggle/kaggle-cli/blob/main/docs/kernels.md#using-secrets-in-kernels).


In [ ]:
# ===== UPLOAD OUTPUT CỦA RUN NÀY LÊN HUGGING FACE =====
HF_UPLOAD_ENABLED = True
HF_REPO_ID = 'AncientWings/AncientWings_R2AI_STAGE3_ds'
HF_SECRET_NAME = 'HF_TOKEN'  # Tạo trong Kaggle: Add-ons > Secrets; không dán token vào code.
HF_UPLOAD_JSONL = True
HF_UPLOAD_ARCHIVE = True
HF_UPLOAD_DIAGNOSTICS = True
HF_UPLOAD_ATTEMPTS = 3


def upload_current_run_to_hf():
    import hashlib
    import json
    from datetime import datetime, timezone
    from pathlib import Path
    import subprocess
    import sys
    import time
    from urllib.parse import quote

    if not HF_UPLOAD_ENABLED:
        print('Đã tắt upload Hugging Face; output vẫn được lưu trong Kaggle Output.')
        return {'status': 'disabled'}

    run_dir = Path(RUN_DIR)
    manifest_path = run_dir / 'notebook_manifest.json'
    if not manifest_path.is_file():
        raise RuntimeError('Chạy mục 7–9 để xuất corpus và đóng ZIP trước khi upload.')
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    if (manifest.get('id_from_inclusive'), manifest.get('id_to_inclusive')) != (ID_FROM, ID_TO):
        raise RuntimeError('Manifest không khớp khoảng ID hiện tại; chạy lại mục 7–9.')
    if manifest.get('dataset_revision') != DATASET_REVISION:
        raise RuntimeError('Manifest không khớp revision nguồn; chạy lại mục 7–9.')
    if not manifest.get('complete') or manifest.get('pending', 1) != 0:
        print(f"Chưa upload: run còn {manifest.get('pending', '?')} URL pending. "
              'Chạy tiếp mục 6, rồi chạy lại mục 7–10.')
        return {'status': 'incomplete', 'run_name': run_dir.name}
    if type(HF_UPLOAD_ATTEMPTS) is not int or HF_UPLOAD_ATTEMPTS < 1:
        raise ValueError('HF_UPLOAD_ATTEMPTS phải là số nguyên dương.')

    run_name = run_dir.name
    files = [(manifest_path, f'manifests/{run_name}/notebook_manifest.json')]
    if manifest.get('accepted', 0) > 0:
        files.append((run_dir / 'corpus.parquet', f'corpus/{run_name}.parquet'))
        if HF_UPLOAD_JSONL:
            files.append((run_dir / 'corpus.jsonl', f'corpus_jsonl/{run_name}.jsonl'))
    else:
        print('Run không có tài liệu accepted; chỉ upload manifest và các ZIP đã chọn.')
    if HF_UPLOAD_ARCHIVE:
        files.append((Path(WORKDIR) / f'{run_name}.zip', f'archives/{run_name}.zip'))
    if HF_UPLOAD_DIAGNOSTICS:
        files.append((Path(WORKDIR) / f'{run_name}_diagnostics.zip',
                      f'diagnostics/{run_name}_diagnostics.zip'))
    for local_path, _ in files:
        if not local_path.is_file() or local_path.stat().st_size == 0:
            raise RuntimeError(f'Thiếu output hoặc file rỗng: {local_path.name}. Chạy lại mục 7–9.')

    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret(HF_SECRET_NAME)
    except Exception:
        raise RuntimeError(f'Tạo và bật Kaggle Secret {HF_SECRET_NAME!r} cho notebook này; '
                           'token cần quyền ghi vào dataset đích.') from None
    if not isinstance(token, str) or not token.strip():
        raise RuntimeError(f'Kaggle Secret {HF_SECRET_NAME!r} đang rỗng.')
    token = token.strip()

    try:
        from huggingface_hub import HfApi, CommitOperationAdd
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                        'huggingface_hub>=0.34,<3'], check=True)
        from huggingface_hub import HfApi, CommitOperationAdd
    api = HfApi(token=token)  # Không login(), không lưu token vào file hay biến môi trường.
    try:
        api.repo_info(repo_id=HF_REPO_ID, repo_type='dataset')
    except Exception as error:
        status = getattr(getattr(error, 'response', None), 'status_code', None)
        raise RuntimeError(f'Không truy cập được dataset {HF_REPO_ID} (HTTP {status or "?"}). '
                           'Kiểm tra Internet, repo đích và quyền của HF_TOKEN.') from None

    file_records = []
    for local_path, remote_path in files:
        digest = hashlib.sha256()
        with local_path.open('rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(block)
        file_records.append({'path_in_repo': remote_path, 'bytes': local_path.stat().st_size,
                             'sha256': digest.hexdigest()})
    upload_manifest = {
        'format_version': 1,
        'source_notebook': 'crawl_corpus_kaggle_hf_upload.ipynb',
        'repo_id': HF_REPO_ID, 'run_name': run_name,
        'id_from_inclusive': ID_FROM, 'id_to_inclusive': ID_TO,
        'dataset_revision': manifest['dataset_revision'],
        'source_sha256': manifest['source_sha256'],
        'complete': True, 'corpus_policy': 'accepted_only',
        'counts': {key: manifest[key] for key in ('selected', 'completed', 'pending', 'accepted')},
        'prepared_at_utc': datetime.now(timezone.utc).isoformat(),
        'files': file_records,
    }
    upload_manifest_path = run_dir / 'hf_upload_manifest.json'
    upload_manifest_path.write_text(json.dumps(upload_manifest, ensure_ascii=False, indent=2) + '\n',
                                    encoding='utf-8')
    files.append((upload_manifest_path, f'manifests/{run_name}/upload_manifest.json'))
    print(f'Dataset đích: {HF_REPO_ID} | Run: {run_name} | Accepted: {manifest["accepted"]}')
    for local_path, remote_path in files:
        print(f'  {remote_path} ({local_path.stat().st_size / 1024**2:.2f} MiB)')

    for attempt in range(1, HF_UPLOAD_ATTEMPTS + 1):
        try:
            # Một commit cho corpus, manifest và ZIP của cùng run; chỉ thêm/cập nhật các path này.
            # Tạo lại operations khi retry vì SDK có thể thay đổi chúng trong quá trình upload.
            commit = api.create_commit(
                repo_id=HF_REPO_ID, repo_type='dataset',
                operations=[CommitOperationAdd(path_in_repo=remote, path_or_fileobj=str(local))
                            for local, remote in files],
                commit_message=f'Upload crawl {run_name} ({manifest["accepted"]} accepted)',
            )
            break
        except Exception as error:
            status = getattr(getattr(error, 'response', None), 'status_code', None)
            retryable = status is None or status >= 500 or status in (408, 409, 429)
            if not retryable or attempt == HF_UPLOAD_ATTEMPTS:
                raise RuntimeError(f'Upload chưa thành công (HTTP {status or "?"}, '
                                   f'{type(error).__name__}). Kiểm tra quyền ghi và Internet; '
                                   'output local vẫn còn, có thể chạy lại riêng mục 10.') from None
            print(f'Upload lỗi mạng/server (lượt {attempt}/{HF_UPLOAD_ATTEMPTS}); đang thử lại...')
            time.sleep(min(2 ** attempt, 30))

    result = {
        'status': 'uploaded', 'repo_id': HF_REPO_ID, 'run_name': run_name,
        'commit_url': commit.commit_url, 'commit_oid': commit.oid,
        'files': [remote for _, remote in files],
    }
    (run_dir / 'hf_upload_result.json').write_text(
        json.dumps(result, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    print('Upload thành công:', commit.commit_url)
    for _, remote in files:
        print(f'https://huggingface.co/datasets/{HF_REPO_ID}/blob/main/{quote(remote, safe="/")}')
    return result


hf_upload_result = upload_current_run_to_hf()
